# Klasifikasi Sentimen Teks Code-Mixed Indonesia–Inggris pada Komentar YouTube

## Setup

In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"   # harus sebelum import torch

import re, copy, random
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix, cohen_kappa_score)
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
import transformers, sklearn
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

SEED = 42
DATA_PATH = "/content/fix_revisibaru.xlsx"
MODEL_NAME = "indolem/indobert-base-uncased"
MAXLEN = 128
USE_EMOJI_TEXT = False
NAMA_MODEL = ["SVM", "CNN-BiLSTM", "IndoBERT"]

# Setiap bagian memakai hasil tersimpan bila sudah ada; jika belum, model dilatih dan hasilnya disimpan.
RUN_MULTI_SEED = True
RUN_TANPA_5SEED = True
RUN_UJI_5SEED = True
RUN_FOCAL_GAMMA = True
RUN_CNN_BEKU = True
RUN_INDOBERTWEET = True
RUN_LOVO = True
RUN_INDOBERTWEET_5SEED = True
SEED_LIST = [42, 37, 47, 32, 52]
CNN_FREEZE = False
ANOTASI2_PATH = "/content/label_annotator2.xlsx"
VALIDASI_FILTER_PATH = "/content/validasi_filter.xlsx"
OUT_DIR = "hasil_penelitian"
SIMPAN_KE_DRIVE = True   # hasil (termasuk progres pelatihan) disimpan di Google Drive agar aman jika Colab terputus
if SIMPAN_KE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT_DIR = "/content/drive/MyDrive/hasil_penelitian"
    except Exception as e:
        print("Drive tidak dipakai:", e)
os.makedirs(OUT_DIR, exist_ok=True)
print("Folder hasil:", OUT_DIR)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)

def anonim(t):
    return re.sub(r"@\S+", "@user", str(t))

print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print("torch", torch.__version__, "| transformers", transformers.__version__, "| sklearn", sklearn.__version__)

## Pemulihan hasil tersimpan

In [ ]:
import zipfile, glob, shutil

# Nama folder lama di zip -> nama folder di notebook ini
PETA = {"r1_tanpa_5seed": "tanpa_penanganan_5seed", "r2_test_5seed": "uji_5seed", "r3_focal_gamma": "focal_gamma",
        "r4_cnn_beku": "cnn_beku", "r5_indobertweet": "indobertweet", "r5b_tweet_5seed": "indobertweet_5seed", "r6_lovo": "lovo"}

def pulihkan(path_zip):
    """Ekstrak zip hasil tersimpan ke OUT_DIR tanpa menimpa file yang sudah ada."""
    if not os.path.exists(path_zip):
        return
    tmp = "/content/_pulih_" + os.path.basename(path_zip).replace(".zip", "")
    with zipfile.ZipFile(path_zip) as z:
        z.extractall(tmp)
    n = 0
    for d, _, fs in os.walk(tmp):
        for f in fs:
            if not f.endswith(".pkl") and f != "metode_terpilih.json":
                continue
            rel = os.path.relpath(os.path.join(d, f), tmp).split(os.sep)
            rel = [PETA.get(x, x) for x in rel]
            while rel and rel[0] not in set(PETA.values()) | {"imbalance", "multi_seed", "metode_terpilih.json"}:
                rel = rel[1:]
            if not rel:
                continue
            tujuan = os.path.join(OUT_DIR, *rel)
            if not os.path.exists(tujuan):
                os.makedirs(os.path.dirname(tujuan), exist_ok=True)
                shutil.copy2(os.path.join(d, f), tujuan); n += 1
    print(f"{os.path.basename(path_zip)}: {n} file disalin")

for z in sorted(f for f in glob.glob("/content/*.zip") if os.path.isfile(f)):
    pulihkan(z)
for sub in ["imbalance", "multi_seed"] + list(PETA.values()):
    p = os.path.join(OUT_DIR, sub)
    print(f"{sub:24s}: {len(os.listdir(p)) if os.path.exists(p) else 0} file")

## Data dan kesepakatan label dengan LLM

In [ ]:
df = pd.read_excel(DATA_PATH, sheet_name="Data Sentimen")
df = df.rename(columns={df.columns[0]: "url"})     # kolom 'zzzz...' = URL video
df = df[["Komentar", "Sentimen manual", "Sentimen", "Video ID"]].rename(
    columns={"Sentimen manual": "label", "Sentimen": "label_llm"})
df["Komentar"] = df["Komentar"].map(anonim)
print("jumlah komentar mentah:", len(df))

kappa = cohen_kappa_score(df["label_llm"], df["label"])
print(f"Cohen's Kappa LLM vs manual (N={len(df)}): {kappa:.4f}")
print(pd.crosstab(df["label_llm"], df["label"]))

## Preprocessing

In [ ]:
# Kamus slang. HARUS SAMA dengan SLANG di filter.py (dipakai untuk deduplikasi sebelum pelabelan).
# Jika diubah di sini, ubah juga di filter.py lalu jalankan ulang filter.py.
SLANG = {
    "bgt":"banget","bngt":"banget","gk":"tidak","ga":"tidak","gak":"tidak","nggak":"tidak","ngga":"tidak",
    "tdk":"tidak","yg":"yang","dgn":"dengan","dg":"dengan","tp":"tapi","tpi":"tapi","jt":"juta",
    "udh":"sudah","udah":"sudah","sdh":"sudah","blm":"belum","krn":"karena","karna":"karena",
    "utk":"untuk","klo":"kalau","kalo":"kalau","jg":"juga","aja":"saja","sm":"sama","skrg":"sekarang",
}

def clean(t):
    t = str(t).lower().replace("\u200b", "")
    t = re.sub(r"@\S+", " ", t)                 # mention/username
    t = re.sub(r"https?://\S+", " ", t)         # URL
    if USE_EMOJI_TEXT:
        import emoji
        t = emoji.demojize(t, language="id").replace(":", " ").replace("_", " ")
    t = re.sub(r"(.)\1{2,}", r"\1", t)          # asliiii -> asli
    t = re.sub(r"\s+", " ", t).strip()          # newline dan spasi ganda
    return t

def normalize(t):
    return re.sub(r"\b\w+\b", lambda m: SLANG.get(m.group(0), m.group(0)), t)

df["text"] = df["Komentar"].map(clean).map(normalize)
print(df[["Komentar", "text"]].sample(5, random_state=SEED).to_string())

## Pemeriksaan data unik dan label

In [ ]:
# =============================================
# VERIFIKASI DATA UNIK DAN LABEL
# =============================================
# Deduplikasi dilakukan di filter.py SEBELUM pelabelan, sehingga setiap teks unik
# hanya dilabeli satu kali. Sel ini hanya memeriksa; tidak ada baris yang dibuang.
label_valid = ["Negatif", "Netral", "Positif"]
assert df["label"].isin(label_valid).all(), "Ada label manual yang tidak valid."
assert df["text"].str.strip().ne("").all(), "Ada teks kosong setelah preprocessing."

dup = df["text"].duplicated(keep=False)
assert not dup.any(), (
    f"{int(dup.sum())} baris punya teks identik setelah preprocessing. Jalankan filter.py "
    "(dengan deduplikasi) sebelum pelabelan, atau samakan kamus SLANG di filter.py dan notebook.")

label2id = {"Negatif": 0, "Netral": 1, "Positif": 2}
id2label = {v: k for k, v in label2id.items()}
df["y"] = df["label"].map(label2id).astype(int)

print("Jumlah komentar unik:", len(df))
print(pd.DataFrame({"Jumlah": df["label"].value_counts(),
                    "Persentase (%)": (df["label"].value_counts(normalize=True) * 100).round(2)}))
print("\nDISTRIBUSI SENTIMEN PER VIDEO")
display(pd.crosstab(df["Video ID"], df["label"], margins=True))

## Pemeriksaan hasil filter code-mixing

In [ ]:
# =============================================
# VERIFIKASI HASIL FILTER CODE-MIXING
# =============================================
# Penyaringan code-mixing dilakukan SATU KALI di filter.py, sebelum pelabelan.
# Sel ini hanya memastikan dataset yang dilabeli memang hasil filter tersebut,
# lalu menyiapkan kolom bukti bahasa untuk analisis kesalahan dan deskripsi data.
data_asli = pd.read_excel(DATA_PATH, sheet_name="Data Sentimen")

for kol in ["Code Mixing", "Bukti Indonesia", "Bukti Inggris"]:
    assert kol in data_asli.columns, f"Kolom '{kol}' tidak ada. Pastikan data berasal dari output filter.py."
assert data_asli["Code Mixing"].eq("Ya").all(), "Ada komentar bertanda 'Tidak'. Jalankan filter.py tanpa opsi --semua."
for kol in ["Bukti Indonesia", "Bukti Inggris"]:
    assert data_asli[kol].fillna("").astype(str).str.strip().ne("").all(), f"Ada '{kol}' yang kosong."

data_asli["text"] = data_asli["Komentar"].map(anonim).map(clean).map(normalize)
bukti = (data_asli[["text", "Code Mixing", "Bukti Indonesia", "Bukti Inggris"]]
         .drop_duplicates(subset="text"))
audit_final = df[["Komentar", "text", "label", "Video ID"]].merge(
    bukti, on="text", how="left", validate="one_to_one")
assert audit_final["Bukti Inggris"].notna().all(), "Ada komentar di df yang tidak ditemukan di data hasil filter."

print("Komentar terverifikasi code-mixed:", len(audit_final))
print("Bukti Inggris hanya 'gaming':",
      int(audit_final["Bukti Inggris"].astype(str).str.strip().str.lower().eq("gaming").sum()))

## Pembagian data

In [ ]:
SEED = 42

# Dataset: komentar unik hasil audit
assert df["text"].is_unique
assert df["y"].notna().all()

# 80% train, 20% sementara
train, temp = train_test_split(
    df,
    test_size=0.20,
    stratify=df["y"],
    random_state=SEED
)

# Bagian sementara dibagi menjadi
# 10% validation dan 10% test
val, test = train_test_split(
    temp,
    test_size=0.50,
    stratify=temp["y"],
    random_state=SEED
)

# Reset index
train = train.reset_index(drop=True)
val = val.reset_index(drop=True)
test = test.reset_index(drop=True)

print("=" * 55)
print("HASIL STRATIFIED SPLITTING")
print("=" * 55)

print("Total dataset :", len(df))
print("Training      :", len(train))
print("Validation    :", len(val))
print("Testing       :", len(test))

print("\nDistribusi kelas:")

distribusi = pd.DataFrame({
    "Dataset": df["label"].value_counts(),
    "Train": train["label"].value_counts(),
    "Validation": val["label"].value_counts(),
    "Test": test["label"].value_counts()
}).fillna(0).astype(int)

print(distribusi)

# Pemeriksaan data leakage
train_text = set(train["text"])
val_text = set(val["text"])
test_text = set(test["text"])

assert train_text.isdisjoint(val_text)
assert train_text.isdisjoint(test_text)
assert val_text.isdisjoint(test_text)

assert len(train) + len(val) + len(test) == len(df)

print("\nTidak ditemukan teks identik antar-split.")
print("Pembagian dataset berhasil.")

## Tokenisasi

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

lens = np.array([
    len(tok(t)["input_ids"])
    for t in train["text"]
])

print("Median token:", int(np.median(lens)))
print("P95 token:", int(np.percentile(lens, 95)))
print("Maksimum token:", int(lens.max()))
print(f"Terpotong pada MAXLEN={MAXLEN}: {np.mean(lens > MAXLEN):.2%}")


class TextDS(Dataset):
    def __init__(self, d):
        enc = tok(
            d["text"].astype(str).tolist(),
            truncation=True,
            max_length=MAXLEN,
            padding="max_length",
            return_tensors="pt"
        )

        self.ids = enc["input_ids"]
        self.mask = enc["attention_mask"]
        self.y = torch.tensor(d["y"].values, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.ids[i], self.mask[i], self.y[i]


def loaders(bs):
    g = torch.Generator().manual_seed(SEED)

    return (
        DataLoader(TextDS(train), batch_size=bs, shuffle=True, generator=g),
        DataLoader(TextDS(val), batch_size=64),
        DataLoader(TextDS(test), batch_size=64)
    )


cw = compute_class_weight(
    "balanced",
    classes=np.array([0, 1, 2]),
    y=train["y"].values
)

CLASS_W = torch.tensor(cw, dtype=torch.float).to(DEVICE)

print("\nClass weight (Negatif, Netral, Positif):", cw.round(2))

train_loader, val_loader, test_loader = loaders(16)
ids, mask, labels = next(iter(train_loader))

print("\nInput IDs:", ids.shape)
print("Attention Mask:", mask.shape)
print("Labels:", labels.shape)

## Model

In [ ]:
class CNNBiLSTM(nn.Module):
    def __init__(
        self,
        emb_weight,
        pad_id,
        n_filters=128,
        kernel=3,
        hidden=128,
        drop=0.5,
        n_cls=3,
        freeze_emb=False
    ):
        super().__init__()

        assert kernel % 2 == 1, "Kernel harus ganjil"

        self.emb = nn.Embedding.from_pretrained(
            emb_weight.clone(),
            freeze=freeze_emb,
            padding_idx=pad_id
        )

        self.conv = nn.Conv1d(
            emb_weight.size(1),
            n_filters,
            kernel_size=kernel,
            padding=kernel // 2
        )

        self.lstm = nn.LSTM(
            input_size=n_filters,
            hidden_size=hidden,
            batch_first=True,
            bidirectional=True
        )

        self.drop = nn.Dropout(drop)
        self.fc = nn.Linear(hidden * 2, n_cls)

    def forward(self, ids, mask):

        lengths = mask.sum(dim=1).cpu()

        # Embedding
        x = self.emb(ids)*mask.unsqueeze(-1).float()
        x = self.drop(x)

        # CNN
        x = self.conv(x.transpose(1, 2))
        x = torch.relu(x)
        x = x.transpose(1, 2)

        # Hilangkan aktivasi pada padding
        x = x * mask.unsqueeze(-1)

        # Packing: LSTM memproses token valid saja
        packed = nn.utils.rnn.pack_padded_sequence(
            x,
            lengths,
            batch_first=True,
            enforce_sorted=False
        )

        packed_out, _ = self.lstm(packed)

        out, _ = nn.utils.rnn.pad_packed_sequence(
            packed_out,
            batch_first=True,
            total_length=ids.size(1)
        )

        # Masked max-pooling
        out = out.masked_fill(
            mask.unsqueeze(-1) == 0,
            torch.finfo(out.dtype).min
        )

        pooled = out.max(dim=1).values

        return self.fc(self.drop(pooled))


class BertCls(nn.Module):
    # fine-tuning IndoBERT; kepala klasifikasi di atas representasi [CLS]
    def __init__(self, n_cls=3):
        super().__init__()
        self.m = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME,num_labels=n_cls, attn_implementation="eager")
    def forward(self, ids, mask):
        return self.m(input_ids=ids, attention_mask=mask).logits


# Embedding awal CNN-BiLSTM = embedding kata IndoBERT (dipakai di semua bagian)
set_seed(SEED)
bert_for_emb = AutoModel.from_pretrained(MODEL_NAME)
emb_w = bert_for_emb.embeddings.word_embeddings.weight.detach().cpu()
del bert_for_emb
print("Ukuran embedding:", tuple(emb_w.shape))

## Fungsi pelatihan dan evaluasi

In [ ]:
# =============================================
# FUNGSI LATIH, EVALUASI, DAN METODE PENANGANAN IMBALANCE
# =============================================
import torch.nn.functional as F

# Skema yang dibandingkan. "Tanpa penanganan" = cross-entropy biasa (pembanding/baseline).
METODE = ["Tanpa penanganan", "Class weighting", "Random oversampling", "Focal loss"]
# Focal loss adalah modifikasi cross-entropy, sehingga tidak berlaku untuk SVM (hinge loss).
METODE_MODEL = {"SVM": METODE[:3], "CNN-BiLSTM": METODE, "IndoBERT": METODE}
FOCAL_GAMMA = 2.0   # Lin et al. (2017)

HP = {
    "CNN-BiLSTM": dict(bs=32, lr=1e-3, epochs=30, patience=5, wd=1e-4, warmup=False),
    "IndoBERT":   dict(bs=16, lr=2e-5, epochs=10, patience=5, wd=0.01, warmup=True),
}
# SVM linear dengan fitur TF-IDF n-gram kata (1-2) dan n-gram karakter (3-5).
# N-gram karakter membantu ejaan informal dan kata Inggris yang ditulis bervariasi.
SVM_C = 1.0


class SVMTfidf:
    """TF-IDF + LinearSVC. Vektorizer di-fit pada data latih asli (sebelum oversampling)."""
    def __init__(self, seed, class_weight=None):
        self.vec = FeatureUnion([
            ("kata", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
            ("karakter", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True)),
        ])
        self.clf = LinearSVC(C=SVM_C, class_weight=class_weight, random_state=seed, max_iter=10000)

    def fit(self, teks_vec, teks, y):
        self.vec.fit(teks_vec)
        self.clf.fit(self.vec.transform(teks), y)
        return self

    def predict(self, teks):
        return self.clf.predict(self.vec.transform(teks))


@torch.no_grad()
def predict(model, dl):
    if isinstance(model, SVMTfidf):          # dl = DataFrame untuk SVM
        return model.predict(dl["text"].astype(str)), dl["y"].values
    model.eval(); P, Y = [], []
    for ids, mask, y in dl:
        P.append(model(ids.to(DEVICE), mask.to(DEVICE)).argmax(-1).cpu()); Y.append(y)
    return torch.cat(P).numpy(), torch.cat(Y).numpy()


def metrics(y, p):
    pr, rc, f1, _ = precision_recall_fscore_support(y, p, average="macro", zero_division=0)
    return {"Accuracy": accuracy_score(y, p), "Macro Precision": pr, "Macro Recall": rc, "Macro F1": f1}


def f1_per_kelas(y, p, k=3):
    """F1 tiap kelas (0 jika penyebut 0); setara f1_score(average=None, zero_division=0)."""
    cm = np.bincount(k * np.asarray(y) + np.asarray(p), minlength=k * k).reshape(k, k)
    tp = np.diag(cm).astype(float)
    penyebut = cm.sum(0) + cm.sum(1)
    return np.divide(2 * tp, penyebut, out=np.zeros(k), where=penyebut > 0)


class FocalLoss(nn.Module):
    """Focal loss: FL = -(1 - p_t)^gamma * log(p_t). Contoh yang sudah mudah ditebak (p_t besar)
    diberi bobot kecil, sehingga pelatihan lebih fokus pada contoh sulit, yang banyak berasal dari kelas minoritas."""
    def __init__(self, gamma=2.0):
        super().__init__(); self.gamma = gamma
    def forward(self, logits, y):
        logp = F.log_softmax(logits, dim=-1).gather(1, y.unsqueeze(1)).squeeze(1)
        return (-(1 - logp.exp()) ** self.gamma * logp).mean()


def random_oversample(d, seed):
    """Duplikasi acak kelas minoritas sampai jumlahnya sama dengan kelas mayoritas.
    HANYA dipakai pada data latih internal; validasi dan data uji tetap berdistribusi asli."""
    n_max = d["y"].value_counts().max()
    # semua data asli dipertahankan, lalu ditambah duplikat acak sebanyak kekurangannya
    bagian = [pd.concat([g, g.sample(n_max - len(g), replace=True, random_state=seed)])
              for _, g in d.groupby("y")]
    return pd.concat(bagian).sample(frac=1, random_state=seed).reset_index(drop=True)


def buat_loss(metode, y_latih):
    if metode == "Class weighting":
        w = compute_class_weight("balanced", classes=np.array([0, 1, 2]), y=y_latih)
        return nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32).to(DEVICE))
    if metode == "Focal loss":
        return FocalLoss(FOCAL_GAMMA)
    return nn.CrossEntropyLoss()          # Tanpa penanganan dan Random oversampling


def fit(model, tr_dl, va_dl, lossf, lr, epochs, patience, weight_decay=0.0, warmup=False):
    model.to(DEVICE)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=weight_decay)
    sched = None
    if warmup:
        total = epochs * len(tr_dl)
        sched = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
    best, best_state, bad = -1, None, 0
    for ep in range(1, epochs + 1):
        model.train(); tot = 0
        for ids, mask, y in tr_dl:
            ids, mask, y = ids.to(DEVICE), mask.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(); loss = lossf(model(ids, mask), y); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            if sched: sched.step()
            tot += loss.item()
        p, y_true = predict(model, va_dl); f1 = metrics(y_true, p)["Macro F1"]
        print(f"epoch {ep:02d} | loss {tot/len(tr_dl):.4f} | val Macro F1 {f1:.4f}")
        if f1 > best: best, best_state, bad = f1, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= patience: print("early stopping"); break
    model.load_state_dict(best_state)
    return model, best


def latih(nama_model, d_latih, d_val, seed, metode):
    """Latih satu model dari awal dengan metode penanganan imbalance tertentu."""
    assert metode in METODE_MODEL[nama_model], (nama_model, metode)
    set_seed(seed)
    if nama_model == "SVM":
        # SVM tidak butuh early stopping; data latih sama dengan model lain, d_val hanya untuk dilaporkan
        cw = "balanced" if metode == "Class weighting" else None    # bobot = n / (k * n_kelas)
        d_fit = random_oversample(d_latih, seed) if metode == "Random oversampling" else d_latih
        model = SVMTfidf(seed, cw).fit(d_latih["text"].astype(str), d_fit["text"].astype(str), d_fit["y"].values)
        p, y_true = predict(model, d_val)
        return model, metrics(y_true, p)["Macro F1"]
    loss_f = buat_loss(metode, d_latih["y"].values)        # bobot kelas dari distribusi asli
    if metode == "Random oversampling":
        d_latih = random_oversample(d_latih, seed)
    hp = HP[nama_model]
    model = CNNBiLSTM(emb_w, tok.pad_token_id, freeze_emb=CNN_FREEZE) if nama_model == "CNN-BiLSTM" else BertCls()
    g = torch.Generator().manual_seed(seed)
    dl_tr = DataLoader(TextDS(d_latih), batch_size=hp["bs"], shuffle=True, generator=g)
    dl_va = DataLoader(TextDS(d_val), batch_size=64)
    return fit(model, dl_tr, dl_va, loss_f, lr=hp["lr"], epochs=hp["epochs"],
               patience=hp["patience"], weight_decay=hp["wd"], warmup=hp["warmup"])


def latih_dan_evaluasi(nama_model, d_latih, d_uji, seed, metode):
    """10% d_latih dipisah untuk early stopping, sisanya dilatih, lalu evaluasi pada d_uji.
    Mengembalikan (pred, y_true, best_val_f1)."""
    d_a, d_v = train_test_split(d_latih, test_size=0.10, stratify=d_latih["y"], random_state=seed)
    d_a, d_v, d_uji = d_a.reset_index(drop=True), d_v.reset_index(drop=True), d_uji.reset_index(drop=True)
    model = None
    try:
        model, best_val = latih(nama_model, d_a, d_v, seed, metode)
        dl_uji = d_uji if nama_model == "SVM" else DataLoader(TextDS(d_uji), batch_size=64)
        pred, y_true = predict(model, dl_uji)
    finally:
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return pred, y_true, best_val


# Cek cepat oversampling pada data latih
_cek = random_oversample(train, SEED)
print("Distribusi latih asli      :", train["label"].value_counts().to_dict())
print("Setelah random oversampling:", _cek["label"].value_counts().to_dict())
assert set(_cek["text"]) == set(train["text"])   # tidak ada teks baru, hanya duplikasi

## Perbandingan skema penanganan ketidakseimbangan kelas

In [ ]:
# =============================================
# CV UNTUK SEMUA SKEMA (dengan resume)
# =============================================
import pickle

data_cv = pd.concat([train, val], ignore_index=True)
assert data_cv["text"].is_unique
assert set(data_cv["text"]).isdisjoint(set(test["text"]))
y_cv = data_cv["y"].values
FOLDS = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(data_cv["text"], data_cv["y"]))

IMB_DIR = os.path.join(OUT_DIR, "imbalance")
os.makedirs(IMB_DIR, exist_ok=True)

baris_imb, oof_imb = [], {}
for m in NAMA_MODEL:
    for metode in METODE_MODEL[m]:
        f_simpan = os.path.join(IMB_DIR, f"{m}__{metode}.pkl".replace(" ", "_"))
        if os.path.exists(f_simpan):                       # resume jika Colab sempat terputus
            with open(f_simpan, "rb") as fh:
                simpan = pickle.load(fh)
            print(f"[dimuat] {m} | {metode}")
        else:
            oof = np.full(len(data_cv), -1, dtype=int); baris = []
            for fold, (i_tr, i_ev) in enumerate(FOLDS, start=1):
                if m != "SVM":
                    print(f"\n{'=' * 20} {m} | {metode} | fold {fold}/5 {'=' * 20}")
                pred, y_true, bv = latih_dan_evaluasi(m, data_cv.iloc[i_tr], data_cv.iloc[i_ev],
                                                      seed=SEED + fold, metode=metode)
                oof[i_ev] = pred
                sk, f1k = metrics(y_true, pred), f1_per_kelas(y_true, pred)
                baris.append({"Model": m, "Metode": metode, "Fold": fold, **sk,
                              "F1 Negatif": f1k[0], "F1 Netral": f1k[1], "F1 Positif": f1k[2],
                              "Best Val F1": bv})
                print(f"{m} | {metode} | Fold {fold}: Accuracy={sk['Accuracy']:.4f} | Macro F1={sk['Macro F1']:.4f}")
            assert (oof >= 0).all()
            simpan = {"baris": baris, "oof": oof}
            with open(f_simpan, "wb") as fh:
                pickle.dump(simpan, fh)
        baris_imb += simpan["baris"]
        oof_imb[(m, metode)] = simpan["oof"]

hasil_imb = pd.DataFrame(baris_imb)
hasil_imb.to_csv(f"{OUT_DIR}/imbalance_per_fold.csv", index=False)
print("\nSelesai:", hasil_imb.groupby(["Model", "Metode"]).size().to_dict())

In [ ]:
# =============================================
# REKAP PERBANDINGAN SKEMA + PILIH SKEMA TERBAIK
# =============================================
import matplotlib.pyplot as plt
import json
from scipy import stats

KOL = ["Accuracy", "Macro Precision", "Macro Recall", "Macro F1"]
baris = []
for m in NAMA_MODEL:
    dasar = hasil_imb[(hasil_imb.Model == m) & (hasil_imb.Metode == "Tanpa penanganan")].sort_values("Fold")
    for metode in METODE_MODEL[m]:
        d = hasil_imb[(hasil_imb.Model == m) & (hasil_imb.Metode == metode)].sort_values("Fold")
        oof = oof_imb[(m, metode)]
        f1k = f1_per_kelas(y_cv, oof)
        r = {"Model": m, "Metode": metode}
        for k in KOL:
            r[k] = d[k].mean(); r[k + " SD"] = d[k].std()
        r.update({"F1 Negatif OOF": f1k[0], "F1 Netral OOF": f1k[1], "F1 Positif OOF": f1k[2],
                  "Recall Negatif OOF": ((oof == 0) & (y_cv == 0)).sum() / (y_cv == 0).sum()})
        # Uji-t terkoreksi Nadeau-Bengio (5 fold) terhadap tanpa penanganan, pada Macro F1
        if metode != "Tanpa penanganan":
            selisih = d["Macro F1"].values - dasar["Macro F1"].values
            k_ = len(selisih); n_uji = len(data_cv) / k_; n_lat = len(data_cv) - n_uji
            var = selisih.var(ddof=1)
            t = selisih.mean() / np.sqrt((1 / k_ + n_uji / n_lat) * var) if var > 0 else np.nan
            r["Selisih Macro F1 vs tanpa"] = selisih.mean()
            r["p (t terkoreksi)"] = 2 * stats.t.sf(abs(t), df=k_ - 1) if var > 0 else np.nan
        baris.append(r)
tab_imb = pd.DataFrame(baris)
tab_imb.to_csv(f"{OUT_DIR}/tabel_perbandingan_imbalance.csv", index=False)

# Tampilan ringkas: mean ± SD antar-fold
tampil = tab_imb[["Model", "Metode"]].copy()
for k in KOL:
    tampil[k] = [f"{a:.4f} ± {b:.4f}" for a, b in zip(tab_imb[k], tab_imb[k + " SD"])]
for k in ["F1 Negatif OOF", "Recall Negatif OOF", "Selisih Macro F1 vs tanpa", "p (t terkoreksi)"]:
    tampil[k] = tab_imb[k].round(4)
print("PERBANDINGAN METODE PENANGANAN KETIDAKSEIMBANGAN KELAS (5-fold CV, mean ± SD)")
display(tampil)

# Pilih metode dengan rata-rata Macro F1 tertinggi per model (dipilih dari CV; data test tidak disentuh)
METODE_TERBAIK = {m: tab_imb[tab_imb.Model == m].sort_values("Macro F1", ascending=False).iloc[0]["Metode"]
                  for m in NAMA_MODEL}
# Untuk memaksa metode tertentu, timpa di sini, mis.: METODE_TERBAIK["IndoBERT"] = "Focal loss"
print("\nMetode terpilih:", METODE_TERBAIK)
with open(f"{OUT_DIR}/metode_terpilih.json", "w") as fh:
    json.dump(METODE_TERBAIK, fh, indent=2)

# Grafik Macro F1 dan F1 Negatif per metode
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x = np.arange(len(METODE)); w = 0.8 / len(NAMA_MODEL)
for ax, (kol, judul) in zip(axes, [("Macro F1", "Macro F1 (mean ± SD fold)"), ("F1 Negatif OOF", "F1 kelas Negatif (OOF)")]):
    for j, m in enumerate(NAMA_MODEL):
        d = tab_imb[tab_imb.Model == m].set_index("Metode").reindex(METODE)   # SVM: focal loss kosong
        err = d[kol + " SD"] if kol + " SD" in d else None
        ax.bar(x + (j - (len(NAMA_MODEL) - 1) / 2) * w, d[kol], w, yerr=err, capsize=3, label=m)
    ax.set_xticks(x); ax.set_xticklabels(METODE, rotation=15); ax.set_title(judul); ax.set_ylim(0, 1)
axes[0].legend()
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/perbandingan_imbalance.png", dpi=300); plt.show()

## Hasil cross-validation skema terpilih

In [ ]:
# =============================================
# HASIL 5-FOLD CV UNTUK SKEMA TERPILIH
# =============================================
# Fold dan seed sama dengan perbandingan skema, jadi hasil CV metode terpilih tidak perlu dilatih ulang.
hasil_cv = pd.concat([
    hasil_imb[(hasil_imb.Model == m) & (hasil_imb.Metode == METODE_TERBAIK[m])] for m in NAMA_MODEL
]).drop(columns="Metode").reset_index(drop=True)
prediksi_oof = {m: oof_imb[(m, METODE_TERBAIK[m])] for m in NAMA_MODEL}
print("Metode:", METODE_TERBAIK)
print(hasil_cv.groupby("Model")[["Accuracy", "Macro F1"]].agg(["mean", "std"]).round(4))

In [ ]:
# ============================================
# REKAP HASIL 5-FOLD
# ============================================

print("=" * 65)
print("HASIL SETIAP FOLD")
print("=" * 65)

display(
    hasil_cv[
        ["Model", "Fold", "Accuracy", "Macro F1",
         "F1 Negatif", "F1 Netral", "F1 Positif"]
    ].round(4)
)

# Rata-rata dan standar deviasi
kolom_metrik = [
    "Accuracy",
    "Macro Precision",
    "Macro Recall",
    "Macro F1",
    "F1 Negatif",
    "F1 Netral",
    "F1 Positif"
]

rata = hasil_cv.groupby("Model")[kolom_metrik].mean()
std = hasil_cv.groupby("Model")[kolom_metrik].std()

ringkasan_cv = pd.DataFrame(index=rata.index)

for kolom in kolom_metrik:
    ringkasan_cv[kolom] = [
        f"{rata.loc[m, kolom]:.4f} ± {std.loc[m, kolom]:.4f}"
        for m in rata.index
    ]

print("\n" + "=" * 65)
print("RATA-RATA ± STANDARD DEVIATION")
print("=" * 65)

display(ringkasan_cv)

print("\n" + "=" * 65)
print("CLASSIFICATION REPORT OUT-OF-FOLD")
print("=" * 65)

for nama_model, pred in prediksi_oof.items():
    print("\nMODEL:", nama_model)

    print(
        classification_report(
            data_cv["y"].values,
            pred,
            labels=[0, 1, 2],
            target_names=[
                id2label[i] for i in range(3)
            ],
            digits=4,
            zero_division=0
        )
    )

## Fungsi statistik

In [ ]:
# =============================================
# FUNGSI BANTU STATISTIK
# =============================================
import pickle, json
from scipy import stats

N_UJI = len(data_cv) / 5             # ukuran fold evaluasi (n2)
N_LATIH = len(data_cv) - N_UJI       # ukuran data latih fold luar (n1)
print(f"n1 (latih) = {N_LATIH:.0f}, n2 (uji) = {N_UJI:.0f}, rasio n2/n1 = {N_UJI / N_LATIH:.4f}")


def uji_terkoreksi(selisih, n_seed):
    """Selisih per (seed, fold). n_seed = 1 -> Nadeau-Bengio (df 4); 5 -> Bouckaert-Frank (df 24)."""
    d = np.asarray(selisih, dtype=float); J = len(d)
    var = d.var(ddof=1)
    se = np.sqrt((1 / J + N_UJI / N_LATIH) * var)
    if se == 0:
        return dict(Selisih=d.mean(), t=np.nan, df=J - 1, p=np.nan, IK_bawah=d.mean(), IK_atas=d.mean())
    t = d.mean() / se
    tk = stats.t.ppf(0.975, J - 1)
    return dict(Selisih=d.mean(), t=t, df=J - 1, p=2 * stats.t.sf(abs(t), J - 1),
                IK_bawah=d.mean() - tk * se, IK_atas=d.mean() + tk * se)


def holm(p):
    """p-value terkoreksi Holm (urutan hasil sama dengan input)."""
    p = np.asarray(p, dtype=float); m = len(p); urut = np.argsort(p)
    adj = np.empty(m); maks = 0
    for r, i in enumerate(urut):
        maks = max(maks, (m - r) * p[i]); adj[i] = min(1.0, maks)
    return adj


def ringkas_kelas(y, p):
    f1k = f1_per_kelas(y, p)
    return {"F1 Negatif": f1k[0], "F1 Netral": f1k[1], "F1 Positif": f1k[2],
            "Recall Negatif": ((p == 0) & (y == 0)).sum() / max((y == 0).sum(), 1)}


def jalankan_cv(m, metode, seed_split, folds=None, simpan=None):
    """5-fold CV satu konfigurasi; hasil disimpan ke file pkl agar bisa dilanjutkan."""
    if simpan and os.path.exists(simpan):
        with open(simpan, "rb") as fh:
            print(f"[dimuat] {os.path.basename(simpan)}"); return pickle.load(fh)
    if folds is None:
        folds = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=seed_split)
                     .split(data_cv["text"], data_cv["y"]))
    oof = np.full(len(data_cv), -1, dtype=int); baris = []
    for fold, (i_tr, i_ev) in enumerate(folds, start=1):
        if m != "SVM":
            print(f"\n--- {m} | {metode} | seed {seed_split} | fold {fold}/5 ---")
        pred, y_true, _ = latih_dan_evaluasi(m, data_cv.iloc[i_tr], data_cv.iloc[i_ev],
                                             seed=seed_split + fold, metode=metode)
        oof[i_ev] = pred
        baris.append({"Seed": seed_split, "Model": m, "Metode": metode, "Fold": fold,
                      **metrics(y_true, pred), **ringkas_kelas(y_true, pred)})
    hasil = {"baris": baris, "oof": oof}
    if simpan:
        with open(simpan, "wb") as fh: pickle.dump(hasil, fh)
    return hasil

## Cross-validation lima seed: skema terpilih

In [ ]:
# =============================================
# CROSS-VALIDATION LIMA SEED: SKEMA TERPILIH
# =============================================
# Setiap seed memakai pembagian fold berbeda. Seed 42 identik dengan perbandingan skema, jadi diambil dari sana.
# Hasil tiap (seed, model) disimpan ke Drive; jika Colab terputus, jalankan ulang dan bagian yang selesai dimuat.
import pickle
MS_DIR = os.path.join(OUT_DIR, "multi_seed")
os.makedirs(MS_DIR, exist_ok=True)

if RUN_MULTI_SEED:
    baris_ms = []
    for s in SEED_LIST:
        for m in NAMA_MODEL:
            f_simpan = os.path.join(MS_DIR, f"seed{s}__{m}.pkl")
            if s == SEED:
                d = hasil_imb[(hasil_imb.Model == m) & (hasil_imb.Metode == METODE_TERBAIK[m])]
                baris = [{"Seed": s, "Model": m, "Fold": int(r.Fold), "Macro F1": r["Macro F1"],
                          "Accuracy": r["Accuracy"], "F1 Negatif": r["F1 Negatif"]} for _, r in d.iterrows()]
            elif os.path.exists(f_simpan):
                with open(f_simpan, "rb") as fh:
                    baris = pickle.load(fh)
                print(f"[dimuat] seed {s} | {m}")
            else:
                baris = []
                skf_s = StratifiedKFold(n_splits=5, shuffle=True, random_state=s)
                for fold, (i_tr, i_ev) in enumerate(skf_s.split(data_cv["text"], data_cv["y"]), start=1):
                    if m != "SVM":
                        print(f"\n--- seed {s} | {m} ({METODE_TERBAIK[m]}) | fold {fold}/5 ---")
                    pred, y_true, _ = latih_dan_evaluasi(m, data_cv.iloc[i_tr], data_cv.iloc[i_ev],
                                                         seed=s + fold, metode=METODE_TERBAIK[m])
                    sk = metrics(y_true, pred)
                    baris.append({"Seed": s, "Model": m, "Fold": fold, "Macro F1": sk["Macro F1"],
                                  "Accuracy": sk["Accuracy"], "F1 Negatif": f1_per_kelas(y_true, pred)[0]})
                with open(f_simpan, "wb") as fh:
                    pickle.dump(baris, fh)
            baris_ms += baris
    ms = pd.DataFrame(baris_ms)
    ms.to_csv(f"{OUT_DIR}/cv_multi_seed_per_fold.csv", index=False)

    per_seed = ms.groupby(["Seed", "Model"])[["Macro F1", "Accuracy", "F1 Negatif"]].mean().unstack("Model")
    print("RATA-RATA PER SEED"); display(per_seed.round(4))
    per_seed.to_csv(f"{OUT_DIR}/cv_multi_seed_per_seed.csv")
    ringkas_ms = ms.groupby(["Seed", "Model"])[["Macro F1", "Accuracy", "F1 Negatif"]].mean() \
                   .groupby("Model").agg(["mean", "std"])
    print("RATA-RATA ± SD ANTAR-SEED"); display(ringkas_ms.round(4))
    ringkas_ms.to_csv(f"{OUT_DIR}/cv_multi_seed_ringkasan.csv")

    piv = ms.pivot_table(index=["Seed", "Fold"], columns="Model", values=["Macro F1", "Accuracy"])
    k_fold, r_seed = 5, ms["Seed"].nunique()
    n_uji = len(data_cv) / k_fold
    n_latih = len(data_cv) - n_uji
    PASANGAN = [("IndoBERT", "CNN-BiLSTM"), ("IndoBERT", "SVM"), ("CNN-BiLSTM", "SVM")]
    baris_t = []
    for a_, b_ in PASANGAN:
        for met in ["Macro F1", "Accuracy"]:
            d = (piv[(met, a_)] - piv[(met, b_)]).values
            menang_fold = int((d > 0).sum())
            menang_seed = int((piv[(met, a_)].groupby("Seed").mean() > piv[(met, b_)].groupby("Seed").mean()).sum())
            # Uji-t terkoreksi untuk CV berulang (Bouckaert & Frank, 2004)
            t = d.mean() / np.sqrt((1 / (k_fold * r_seed) + n_uji / n_latih) * d.var(ddof=1))
            p = 2 * stats.t.sf(abs(t), df=k_fold * r_seed - 1)
            print(f"{a_} vs {b_} | {met}: selisih = {d.mean():.4f} (SD {d.std(ddof=1):.4f}); {a_} unggul pada "
                  f"{menang_seed}/{r_seed} seed dan {menang_fold}/{len(d)} fold; t terkoreksi = {t:.3f}, p = {p:.4f}")
            baris_t.append({"Pasangan": f"{a_} vs {b_}", "Metrik": met, "Selisih": d.mean(),
                            "SD selisih": d.std(ddof=1), "Unggul seed": menang_seed, "Unggul fold": menang_fold,
                            "t": t, "p": p})
    tab_t = pd.DataFrame(baris_t)
    display(tab_t.round(4))
    tab_t.to_csv(f"{OUT_DIR}/uji_t_terkoreksi_multi_seed.csv", index=False)
else:
    print("Bagian multi-seed dilewati (RUN_MULTI_SEED = False).")

## Cross-validation lima seed: tanpa penanganan

In [ ]:
# =============================================
# CROSS-VALIDATION LIMA SEED: TANPA PENANGANAN
# =============================================
TANPA_DIR = os.path.join(OUT_DIR, "tanpa_penanganan_5seed"); os.makedirs(TANPA_DIR, exist_ok=True)
if RUN_TANPA_5SEED:
    baris_tanpa, oof_tanpa = [], {}
    for s in SEED_LIST:
        for m in NAMA_MODEL:
            if s == SEED:
                d = hasil_imb[(hasil_imb.Model == m) & (hasil_imb.Metode == "Tanpa penanganan")]
                oof = oof_imb[(m, "Tanpa penanganan")]
                bs = []
                for (_, r), (i_tr, i_ev) in zip(d.sort_values("Fold").iterrows(), FOLDS):
                    bs.append({"Seed": s, "Model": m, "Metode": "Tanpa penanganan", "Fold": int(r.Fold),
                               **{k: r[k] for k in ["Accuracy", "Macro Precision", "Macro Recall", "Macro F1"]},
                               **ringkas_kelas(y_cv[i_ev], oof[i_ev])})
                h = {"baris": bs, "oof": oof}
            else:
                h = jalankan_cv(m, "Tanpa penanganan", s, simpan=os.path.join(TANPA_DIR, f"seed{s}__{m}.pkl"))
            baris_tanpa += h["baris"]; oof_tanpa[(s, m)] = h["oof"]
    tanpa5 = pd.DataFrame(baris_tanpa)
    tanpa5.to_csv(f"{OUT_DIR}/tanpa_penanganan_5seed_per_fold.csv", index=False)

    # Ringkasan: rata-rata antar-seed (rata-rata fold per seed, lalu mean ± SD antar-seed)
    KOLR = ["Macro F1", "Accuracy", "F1 Negatif", "Recall Negatif"]
    ring_tanpa5 = tanpa5.groupby(["Model", "Seed"])[KOLR].mean().groupby("Model").agg(["mean", "std"])
    print("TANPA PENANGANAN, 5 SEED (mean ± SD antar-seed)"); display(ring_tanpa5.round(4))
    ring_tanpa5.to_csv(f"{OUT_DIR}/tanpa_penanganan_ringkasan.csv")

    # (a) Perbandingan model dengan skema tetap "Tanpa penanganan" (Bouckaert-Frank, df 24) + Holm
    piv = tanpa5.pivot_table(index=["Seed", "Fold"], columns="Model", values="Macro F1")
    PASANGAN = [("IndoBERT", "CNN-BiLSTM"), ("IndoBERT", "SVM"), ("SVM", "CNN-BiLSTM")]
    rows = []
    for a_, b_ in PASANGAN:
        d = (piv[a_] - piv[b_]).values
        rows.append({"Pasangan": f"{a_} vs {b_}", "Unggul seed": int((piv[a_].groupby("Seed").mean() >
                     piv[b_].groupby("Seed").mean()).sum()), "Unggul fold": int((d > 0).sum()),
                     **uji_terkoreksi(d, 5)})
    t_model = pd.DataFrame(rows); t_model["p Holm"] = holm(t_model["p"])
    print("\nPERBANDINGAN MODEL, SKEMA TETAP TANPA PENANGANAN (Macro F1, 5 seed × 5 fold)")
    display(t_model.round(4)); t_model.to_csv(f"{OUT_DIR}/uji_model_tanpa_penanganan.csv", index=False)

    # (b) Metode terpilih vs tanpa penanganan per model, 5 seed + Holm
    ms = pd.read_csv(f"{OUT_DIR}/cv_multi_seed_per_fold.csv")      # hasil CV lima seed metode terpilih
    rows = []
    for m in NAMA_MODEL:
        a = ms[ms.Model == m].set_index(["Seed", "Fold"])["Macro F1"]
        b = tanpa5[tanpa5.Model == m].set_index(["Seed", "Fold"])["Macro F1"]
        idx = a.index.intersection(b.index); assert len(idx) == 25, (m, len(idx))
        d = (a.loc[idx] - b.loc[idx]).values
        rows.append({"Model": m, "Metode terpilih": METODE_TERBAIK[m], "Macro F1 terpilih": a.loc[idx].mean(),
                     "Macro F1 tanpa": b.loc[idx].mean(), **uji_terkoreksi(d, 5)})
    t_metode = pd.DataFrame(rows); t_metode["p Holm"] = holm(t_metode["p"])
    print("\nMETODE TERPILIH vs TANPA PENANGANAN (Macro F1, 5 seed × 5 fold)")
    display(t_metode.round(4)); t_metode.to_csv(f"{OUT_DIR}/uji_terpilih_vs_tanpa.csv", index=False)
else:
    print("Bagian ini dilewati.")

## Data uji lima seed

In [ ]:
# =============================================
# DATA UJI DENGAN LIMA SEED
# =============================================
UJI_DIR = os.path.join(OUT_DIR, "uji_5seed"); os.makedirs(UJI_DIR, exist_ok=True)
if RUN_UJI_5SEED:
    y_test = test["y"].values
    baris_uji, pred_uji = [], {}
    for m in NAMA_MODEL:
        for metode in dict.fromkeys(["Tanpa penanganan", METODE_TERBAIK[m]]):
            for s in SEED_LIST:
                f = os.path.join(UJI_DIR, f"{m}__{metode}__seed{s}.pkl".replace(" ", "_"))
                if os.path.exists(f):
                    with open(f, "rb") as fh: h = pickle.load(fh)
                    print(f"[dimuat] {os.path.basename(f)}")
                else:
                    if m != "SVM": print(f"\n--- TEST | {m} | {metode} | seed {s} ---")
                    model, bv = latih(m, train, val, s, metode)
                    dl = test if m == "SVM" else DataLoader(TextDS(test), batch_size=64)
                    pred, _ = predict(model, dl)
                    del model
                    if torch.cuda.is_available(): torch.cuda.empty_cache()
                    h = {"Model": m, "Metode": metode, "Seed": s, "Best Val F1": bv,
                         **metrics(y_test, pred), **ringkas_kelas(y_test, pred), "pred": pred}
                    with open(f, "wb") as fh: pickle.dump(h, fh)
                pred_uji[(m, metode, s)] = h["pred"]
                baris_uji.append({k: v for k, v in h.items() if k != "pred"})
    uji5 = pd.DataFrame(baris_uji)
    uji5.to_csv(f"{OUT_DIR}/uji_5seed_per_seed.csv", index=False)
    KOLT = ["Macro F1", "Accuracy", "F1 Negatif", "Recall Negatif", "Best Val F1"]
    ring_uji = uji5.groupby(["Model", "Metode"])[KOLT].agg(["mean", "std", "min", "max"])
    print("DATA UJI, 5 SEED (mean, SD, min, max)"); display(ring_uji.round(4))
    ring_uji.to_csv(f"{OUT_DIR}/uji_5seed_ringkasan.csv")

    # Selisih CV vs data uji untuk metode terpilih
    ms = pd.read_csv(f"{OUT_DIR}/cv_multi_seed_per_fold.csv")
    cv_mean = ms.groupby("Model")["Macro F1"].mean()
    gap = pd.DataFrame([{"Model": m, "Metode": METODE_TERBAIK[m], "Macro F1 CV (5 seed)": cv_mean[m],
                         "Macro F1 test (5 seed) mean": uji5[(uji5.Model == m) & (uji5.Metode == METODE_TERBAIK[m])]["Macro F1"].mean(),
                         "Macro F1 test SD": uji5[(uji5.Model == m) & (uji5.Metode == METODE_TERBAIK[m])]["Macro F1"].std()}
                        for m in NAMA_MODEL])
    gap["Selisih test - CV"] = gap["Macro F1 test (5 seed) mean"] - gap["Macro F1 CV (5 seed)"]
    print("\nCV vs TEST (metode terpilih)"); display(gap.round(4))
    gap.to_csv(f"{OUT_DIR}/uji_5seed_cv_vs_test.csv", index=False)
    print("Catatan: data uji hanya", len(test), "komentar (Negatif:", int((y_test == 0).sum()),
          "), jadi variasi antar-seed dan ketidakpastian sampling cukup besar.")
else:
    print("Bagian ini dilewati.")

## Sensitivitas focal loss terhadap γ

In [ ]:
# =============================================
# FOCAL LOSS: GAMMA 1, 2, 3
# =============================================
GAMMA_DIR = os.path.join(OUT_DIR, "focal_gamma"); os.makedirs(GAMMA_DIR, exist_ok=True)
if RUN_FOCAL_GAMMA:
    GAMMA_ASLI = FOCAL_GAMMA
    baris_gamma, oof_gamma = [], {}
    for m in ["CNN-BiLSTM", "IndoBERT"]:
        for g in [1.0, 2.0, 3.0]:
            if g == GAMMA_ASLI:
                d = hasil_imb[(hasil_imb.Model == m) & (hasil_imb.Metode == "Focal loss")].sort_values("Fold")
                oof = oof_imb[(m, "Focal loss")]
                bs = [{"Model": m, "Gamma": g, "Fold": int(r.Fold), "Macro F1": r["Macro F1"],
                       "Accuracy": r["Accuracy"]} for _, r in d.iterrows()]
            else:
                FOCAL_GAMMA = g                     # dipakai oleh buat_loss()
                h = jalankan_cv(m, "Focal loss", SEED, folds=FOLDS,
                                simpan=os.path.join(GAMMA_DIR, f"{m}__gamma{g:g}.pkl"))
                FOCAL_GAMMA = GAMMA_ASLI
                oof = h["oof"]
                bs = [{"Model": m, "Gamma": g, "Fold": r["Fold"], "Macro F1": r["Macro F1"],
                       "Accuracy": r["Accuracy"]} for r in h["baris"]]
            baris_gamma += bs; oof_gamma[(m, g)] = oof
    FOCAL_GAMMA = GAMMA_ASLI
    gamma5 = pd.DataFrame(baris_gamma); gamma5.to_csv(f"{OUT_DIR}/focal_gamma_per_fold.csv", index=False)
    rows = []
    for m in ["CNN-BiLSTM", "IndoBERT"]:
        dasar = hasil_imb[(hasil_imb.Model == m) & (hasil_imb.Metode == "Tanpa penanganan")].sort_values("Fold")["Macro F1"].values
        for g in [1.0, 2.0, 3.0]:
            v = gamma5[(gamma5.Model == m) & (gamma5.Gamma == g)].sort_values("Fold")["Macro F1"].values
            u = uji_terkoreksi(v - dasar, 1)
            rows.append({"Model": m, "Gamma": g, "Macro F1 mean": v.mean(), "Macro F1 SD": v.std(ddof=1),
                         **{f"{k} (pooled)": val for k, val in ringkas_kelas(y_cv, oof_gamma[(m, g)]).items()},
                         "Selisih vs tanpa": u["Selisih"], "p vs tanpa": u["p"],
                         "IK bawah": u["IK_bawah"], "IK atas": u["IK_atas"]})
    tab_gamma = pd.DataFrame(rows)
    print("FOCAL LOSS: SENSITIVITAS GAMMA (seed 42, 5-fold CV; uji Nadeau-Bengio vs tanpa penanganan)")
    display(tab_gamma.round(4)); tab_gamma.to_csv(f"{OUT_DIR}/focal_gamma.csv", index=False)
else:
    print("Bagian ini dilewati.")

## Jumlah parameter dan CNN-BiLSTM dengan embedding beku

In [ ]:
# =============================================
# JUMLAH PARAMETER DAN CNN-BiLSTM DENGAN EMBEDDING BEKU
# =============================================
def hitung_param(model):
    tot = sum(p.numel() for p in model.parameters())
    lat = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return tot, lat

baris_par = []
for nama, mdl in [("CNN-BiLSTM (embedding dilatih)", CNNBiLSTM(emb_w, tok.pad_token_id, freeze_emb=False)),
                  ("CNN-BiLSTM (embedding beku)", CNNBiLSTM(emb_w, tok.pad_token_id, freeze_emb=True)),
                  ("IndoBERT", BertCls())]:
    tot, lat = hitung_param(mdl)
    baris_par.append({"Model": nama, "Total parameter": tot, "Parameter dilatih": lat})
    del mdl
_svm = SVMTfidf(SEED, "balanced").fit(data_cv["text"].astype(str), data_cv["text"].astype(str), data_cv["y"].values)
n_fitur = _svm.clf.coef_.shape[1]
baris_par.append({"Model": "SVM (TF-IDF, dilatih pada seluruh data CV)", "Total parameter": 3 * n_fitur + 3,
                  "Parameter dilatih": 3 * n_fitur + 3})
tab_par = pd.DataFrame(baris_par)
print(f"Jumlah fitur TF-IDF SVM: {n_fitur:,} (bobot 3 kelas × fitur + 3 intersep; ukuran berubah sedikit tiap fold)")
display(tab_par); tab_par.to_csv(f"{OUT_DIR}/cnn_beku_jumlah_parameter.csv", index=False)

BEKU_DIR = os.path.join(OUT_DIR, "cnn_beku"); os.makedirs(BEKU_DIR, exist_ok=True)
if RUN_CNN_BEKU:
    rows = []
    for metode in dict.fromkeys(["Tanpa penanganan", METODE_TERBAIK["CNN-BiLSTM"]]):
        CNN_FREEZE = True
        h = jalankan_cv("CNN-BiLSTM", metode, SEED, folds=FOLDS,
                        simpan=os.path.join(BEKU_DIR, f"beku__{metode}.pkl".replace(" ", "_")))
        CNN_FREEZE = False
        beku = np.array([r["Macro F1"] for r in sorted(h["baris"], key=lambda r: r["Fold"])])
        latih_ = hasil_imb[(hasil_imb.Model == "CNN-BiLSTM") & (hasil_imb.Metode == metode)].sort_values("Fold")["Macro F1"].values
        u = uji_terkoreksi(beku - latih_, 1)
        rows.append({"Metode": metode, "Macro F1 beku": beku.mean(), "SD beku": beku.std(ddof=1),
                     "Macro F1 dilatih": latih_.mean(), "SD dilatih": latih_.std(ddof=1),
                     "F1 Negatif beku (pooled)": ringkas_kelas(y_cv, h["oof"])["F1 Negatif"],
                     "F1 Negatif dilatih (pooled)": ringkas_kelas(y_cv, oof_imb[("CNN-BiLSTM", metode)])["F1 Negatif"],
                     "Selisih beku - dilatih": u["Selisih"], "p": u["p"], "IK bawah": u["IK_bawah"], "IK atas": u["IK_atas"]})
    CNN_FREEZE = False
    tab_beku = pd.DataFrame(rows)
    print("CNN-BiLSTM: EMBEDDING BEKU vs DILATIH (seed 42, 5-fold CV, uji Nadeau-Bengio)")
    display(tab_beku.round(4)); tab_beku.to_csv(f"{OUT_DIR}/cnn_beku.csv", index=False)
else:
    print("Bagian ini dilewati.")

## Leave-one-video-out

In [ ]:
# =============================================
# LEAVE-ONE-VIDEO-OUT
# =============================================
LOVO_DIR = os.path.join(OUT_DIR, "lovo"); os.makedirs(LOVO_DIR, exist_ok=True)
if RUN_LOVO:
    semua = pd.concat([train, val, test], ignore_index=True)
    assert "Video ID" in semua.columns and semua["Video ID"].notna().all()
    VIDEO = sorted(semua["Video ID"].unique())
    print("Jumlah komentar per video:", semua["Video ID"].value_counts().to_dict())
    baris_lovo = []
    for m in NAMA_MODEL:
        f = os.path.join(LOVO_DIR, f"{m}.pkl")
        if os.path.exists(f):
            with open(f, "rb") as fh: bs = pickle.load(fh)
            print(f"[dimuat] {os.path.basename(f)}")
        else:
            bs = []
            for k, v in enumerate(VIDEO, start=1):
                d_lat, d_uji = semua[semua["Video ID"] != v], semua[semua["Video ID"] == v]
                if m != "SVM": print(f"\n--- LOVO | {m} | uji video {v} ---")
                pred, y_true, _ = latih_dan_evaluasi(m, d_lat, d_uji, seed=SEED + k, metode=METODE_TERBAIK[m])
                bs.append({"Model": m, "Metode": METODE_TERBAIK[m], "Video uji": v, "n uji": len(d_uji),
                           **metrics(y_true, pred), **ringkas_kelas(y_true, pred)})
            with open(f, "wb") as fh: pickle.dump(bs, fh)
        baris_lovo += bs
    lovo = pd.DataFrame(baris_lovo); lovo.to_csv(f"{OUT_DIR}/lovo_per_video.csv", index=False)
    display(lovo.round(4))
    print("\nRATA-RATA ANTAR-VIDEO"); display(lovo.groupby("Model")[["Macro F1", "Accuracy", "F1 Negatif"]].agg(["mean", "std"]).round(4))
else:
    print("Bagian ini dilewati.")

## IndoBERTweet

In [ ]:
# =============================================
# INDOBERTWEET
# =============================================
TWEET_DIR = os.path.join(OUT_DIR, "indobertweet"); os.makedirs(TWEET_DIR, exist_ok=True)
NAMA_TWEET = "indolem/indobertweet-base-uncased"
def siapkan_tweet():
    """Tokenizer dan kelas model IndoBERTweet."""
    global tok_tw, TextDSTweet, TweetCls
    if "tok_tw" in globals():
        return
    tok_tw = AutoTokenizer.from_pretrained(NAMA_TWEET)

    class TextDSTweet(TextDS):
        def __init__(self, d):
            enc = tok_tw(d["text"].astype(str).str.replace("@user", "@USER").tolist(), truncation=True,
                         max_length=MAXLEN, padding="max_length", return_tensors="pt")
            self.ids, self.mask = enc["input_ids"], enc["attention_mask"]
            self.y = torch.tensor(d["y"].values, dtype=torch.long)

    class TweetCls(nn.Module):
        def __init__(self, n_cls=3):
            super().__init__()
            self.m = AutoModelForSequenceClassification.from_pretrained(NAMA_TWEET, num_labels=n_cls,
                                                                        attn_implementation="eager")
        def forward(self, ids, mask):
            return self.m(input_ids=ids, attention_mask=mask).logits

if RUN_INDOBERTWEET:
    siapkan_tweet()
    rows, baris_tweet = [], []
    for metode in ["Tanpa penanganan", "Focal loss"]:
        f = os.path.join(TWEET_DIR, f"{metode}.pkl".replace(" ", "_"))
        if os.path.exists(f):
            with open(f, "rb") as fh: h = pickle.load(fh)
            print(f"[dimuat] {os.path.basename(f)}")
        else:
            oof = np.full(len(data_cv), -1, dtype=int); bs = []
            for fold, (i_tr, i_ev) in enumerate(FOLDS, start=1):
                print(f"\n--- IndoBERTweet | {metode} | fold {fold}/5 ---")
                s = SEED + fold
                d_a, d_v = train_test_split(data_cv.iloc[i_tr], test_size=0.10, stratify=data_cv.iloc[i_tr]["y"], random_state=s)
                d_a, d_v, d_e = d_a.reset_index(drop=True), d_v.reset_index(drop=True), data_cv.iloc[i_ev].reset_index(drop=True)
                set_seed(s)
                loss_f = buat_loss(metode, d_a["y"].values)
                hp = HP["IndoBERT"]; g = torch.Generator().manual_seed(s)
                model, bv = fit(TweetCls(), DataLoader(TextDSTweet(d_a), batch_size=hp["bs"], shuffle=True, generator=g),
                                DataLoader(TextDSTweet(d_v), batch_size=64), loss_f, lr=hp["lr"], epochs=hp["epochs"],
                                patience=hp["patience"], weight_decay=hp["wd"], warmup=hp["warmup"])
                pred, y_true = predict(model, DataLoader(TextDSTweet(d_e), batch_size=64))
                del model; torch.cuda.empty_cache() if torch.cuda.is_available() else None
                oof[i_ev] = pred
                bs.append({"Model": "IndoBERTweet", "Metode": metode, "Fold": fold, **metrics(y_true, pred)})
            h = {"baris": bs, "oof": oof}
            with open(f, "wb") as fh: pickle.dump(h, fh)
        baris_tweet += h["baris"]
        v = np.array([r["Macro F1"] for r in sorted(h["baris"], key=lambda r: r["Fold"])])
        ib = hasil_imb[(hasil_imb.Model == "IndoBERT") & (hasil_imb.Metode == metode)].sort_values("Fold")["Macro F1"].values
        u = uji_terkoreksi(v - ib, 1)
        rows.append({"Metode": metode, "Macro F1 IndoBERTweet": v.mean(), "SD": v.std(ddof=1),
                     "Macro F1 IndoBERT": ib.mean(), **{f"{k} (pooled)": x for k, x in ringkas_kelas(y_cv, h["oof"]).items()},
                     "Selisih vs IndoBERT": u["Selisih"], "p": u["p"], "IK bawah": u["IK_bawah"], "IK atas": u["IK_atas"]})
    pd.DataFrame(baris_tweet).to_csv(f"{OUT_DIR}/indobertweet_per_fold.csv", index=False)
    tab_tweet = pd.DataFrame(rows)
    print("INDOBERTWEET vs INDOBERT (seed 42, 5-fold CV, uji Nadeau-Bengio)")
    display(tab_tweet.round(4)); tab_tweet.to_csv(f"{OUT_DIR}/indobertweet.csv", index=False)
else:
    print("Bagian ini dilewati.")

In [ ]:
# =============================================
# INDOBERTWEET: LIMA SEED, DATA UJI, LEAVE-ONE-VIDEO-OUT
# =============================================
TWEET5_DIR = os.path.join(OUT_DIR, "indobertweet_5seed"); os.makedirs(TWEET5_DIR, exist_ok=True)


def latih_tweet(d_lat, d_val, s, metode="Tanpa penanganan"):
    siapkan_tweet()
    set_seed(s)
    loss_f = buat_loss(metode, d_lat["y"].values)
    if metode == "Random oversampling":
        d_lat = random_oversample(d_lat, s)
    hp = HP["IndoBERT"]; g = torch.Generator().manual_seed(s)
    return fit(TweetCls(), DataLoader(TextDSTweet(d_lat), batch_size=hp["bs"], shuffle=True, generator=g),
               DataLoader(TextDSTweet(d_val), batch_size=64), loss_f, lr=hp["lr"], epochs=hp["epochs"],
               patience=hp["patience"], weight_decay=hp["wd"], warmup=hp["warmup"])


def evaluasi_tweet(d_lat, d_uji, s, metode="Tanpa penanganan"):
    """Sama dengan latih_dan_evaluasi(): 10% data latih untuk validasi early stopping."""
    d_a, d_v = train_test_split(d_lat, test_size=0.10, stratify=d_lat["y"], random_state=s)
    model, bv = latih_tweet(d_a.reset_index(drop=True), d_v.reset_index(drop=True), s, metode)
    pred, y_true = predict(model, DataLoader(TextDSTweet(d_uji.reset_index(drop=True)), batch_size=64))
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return pred, y_true, bv


if RUN_INDOBERTWEET_5SEED:
    # ---- (a) 5-fold CV x 5 seed
    baris_tw, oof_tw = [], {}
    for s in SEED_LIST:
        if s == SEED:
            with open(os.path.join(OUT_DIR, "indobertweet", "Tanpa_penanganan.pkl"), "rb") as fh:
                h = pickle.load(fh)
            print("[dimuat] seed 42")
            oof = h["oof"]
            bs = [{"Seed": s, "Model": "IndoBERTweet", "Metode": "Tanpa penanganan", "Fold": r["Fold"],
                   **{k: r[k] for k in ["Accuracy", "Macro Precision", "Macro Recall", "Macro F1"]},
                   **ringkas_kelas(y_cv[FOLDS[r["Fold"] - 1][1]], oof[FOLDS[r["Fold"] - 1][1]])} for r in h["baris"]]
        else:
            f = os.path.join(TWEET5_DIR, f"cv_seed{s}.pkl")
            if os.path.exists(f):
                with open(f, "rb") as fh: h = pickle.load(fh)
                print(f"[dimuat] {os.path.basename(f)}")
            else:
                folds_s = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=s).split(data_cv["text"], data_cv["y"]))
                oof = np.full(len(data_cv), -1, dtype=int); bs = []
                for fold, (i_tr, i_ev) in enumerate(folds_s, start=1):
                    print(f"\n--- IndoBERTweet | seed {s} | fold {fold}/5 ---")
                    pred, y_true, _ = evaluasi_tweet(data_cv.iloc[i_tr], data_cv.iloc[i_ev], s + fold)
                    oof[i_ev] = pred
                    bs.append({"Seed": s, "Model": "IndoBERTweet", "Metode": "Tanpa penanganan", "Fold": fold,
                               **metrics(y_true, pred), **ringkas_kelas(y_true, pred)})
                h = {"baris": bs, "oof": oof}
                with open(f, "wb") as fh: pickle.dump(h, fh)
            bs, oof = h["baris"], h["oof"]
        baris_tw += bs; oof_tw[s] = oof
    tw = pd.DataFrame(baris_tw); tw.to_csv(f"{OUT_DIR}/indobertweet_5seed_cv_per_fold.csv", index=False)

    # gabung dengan tiga model lain (tanpa penanganan, 5 seed)
    tanpa_all = pd.read_csv(f"{OUT_DIR}/tanpa_penanganan_5seed_per_fold.csv")
    semua4 = pd.concat([tanpa_all, tw], ignore_index=True)
    KOLR = ["Macro F1", "Accuracy", "F1 Negatif", "Recall Negatif"]
    ring4 = semua4.groupby(["Model", "Seed"])[KOLR].mean().groupby("Model").agg(["mean", "std"])
    print("EMPAT MODEL, TANPA PENANGANAN, 5 SEED (mean ± SD antar-seed)"); display(ring4.round(4))
    ring4.to_csv(f"{OUT_DIR}/indobertweet_5seed_ringkasan_4model.csv")

    piv4 = semua4.pivot_table(index=["Seed", "Fold"], columns="Model", values="Macro F1")
    assert len(piv4.dropna()) == 25
    URUT = ["IndoBERTweet", "IndoBERT", "SVM", "CNN-BiLSTM"]
    rows = []
    for i_, a_ in enumerate(URUT):
        for b_ in URUT[i_ + 1:]:
            d = (piv4[a_] - piv4[b_]).values
            rows.append({"Pasangan": f"{a_} vs {b_}",
                         "Unggul seed": int((piv4[a_].groupby("Seed").mean() > piv4[b_].groupby("Seed").mean()).sum()),
                         "Unggul fold": int((d > 0).sum()), **uji_terkoreksi(d, 5)})
    t4 = pd.DataFrame(rows); t4["p Holm"] = holm(t4["p"])
    t4["Signifikan (Holm, α=0,05)"] = np.where(t4["p Holm"] < 0.05, "Ya", "Tidak")
    print("\nPERBANDINGAN 4 MODEL (Macro F1, 5 seed × 5 fold, Bouckaert-Frank, Holm atas 6 pasangan)")
    display(t4.round(4)); t4.to_csv(f"{OUT_DIR}/indobertweet_5seed_uji_4model.csv", index=False)

    # ---- (b) data uji, 5 seed
    y_test = test["y"].values; baris_te = []
    for s in SEED_LIST:
        f = os.path.join(TWEET5_DIR, f"test_seed{s}.pkl")
        if os.path.exists(f):
            with open(f, "rb") as fh: h = pickle.load(fh)
            print(f"[dimuat] {os.path.basename(f)}")
        else:
            print(f"\n--- TEST | IndoBERTweet | seed {s} ---")
            model, bv = latih_tweet(train, val, s)
            pred, _ = predict(model, DataLoader(TextDSTweet(test), batch_size=64))
            del model
            if torch.cuda.is_available(): torch.cuda.empty_cache()
            h = {"Model": "IndoBERTweet", "Metode": "Tanpa penanganan", "Seed": s, "Best Val F1": bv,
                 **metrics(y_test, pred), **ringkas_kelas(y_test, pred), "pred": pred}
            with open(f, "wb") as fh: pickle.dump(h, fh)
        baris_te.append({k: v for k, v in h.items() if k != "pred"})
    te = pd.DataFrame(baris_te); te.to_csv(f"{OUT_DIR}/indobertweet_5seed_test_per_seed.csv", index=False)
    print("\nDATA UJI IndoBERTweet, 5 SEED"); display(te.round(4))
    display(te[["Macro F1", "Accuracy", "F1 Negatif", "Recall Negatif"]].agg(["mean", "std", "min", "max"]).round(4))

    # ---- (c) leave-one-video-out
    semua = pd.concat([train, val, test], ignore_index=True)
    VIDEO = sorted(semua["Video ID"].unique())
    f = os.path.join(TWEET5_DIR, "lovo.pkl")
    if os.path.exists(f):
        with open(f, "rb") as fh: bs = pickle.load(fh)
        print("[dimuat] lovo.pkl")
    else:
        bs = []
        for k, v in enumerate(VIDEO, start=1):
            d_lat, d_uji = semua[semua["Video ID"] != v], semua[semua["Video ID"] == v]
            print(f"\n--- LOVO | IndoBERTweet | uji video {v} ---")
            pred, y_true, _ = evaluasi_tweet(d_lat, d_uji, SEED + k)
            bs.append({"Model": "IndoBERTweet", "Metode": "Tanpa penanganan", "Video uji": v, "n uji": len(d_uji),
                       **metrics(y_true, pred), **ringkas_kelas(y_true, pred)})
        with open(f, "wb") as fh: pickle.dump(bs, fh)
    lv = pd.DataFrame(bs); lv.to_csv(f"{OUT_DIR}/indobertweet_5seed_lovo.csv", index=False)
    print("\nLEAVE-ONE-VIDEO-OUT IndoBERTweet"); display(lv.round(4))
    print("Rata-rata Macro F1 antar-video:", round(lv["Macro F1"].mean(), 4))
else:
    print("Bagian ini dilewati.")

## Rekap per fold dan ekspor prediksi

In [ ]:
# =============================================
# REKAP PER FOLD DAN EKSPOR PREDIKSI
# =============================================
import json, shutil
import matplotlib.pyplot as plt
from scipy import stats

NAMA_KELAS = [id2label[i] for i in range(3)]

for nama in ("data_cv", "hasil_cv", "prediksi_oof"):
    assert nama in globals(), f"'{nama}' belum ada. Jalankan sel hasil CV skema terpilih terlebih dahulu."
assert set(prediksi_oof) == set(NAMA_MODEL), "Nama model di prediksi_oof tidak sesuai."

y_cv = data_cv["y"].values
K_FOLD = hasil_cv["Fold"].nunique()


def f1_per_kelas(y, p, k=3):
    """F1 tiap kelas dari confusion matrix (0 jika penyebut 0); setara f1_score(average=None, zero_division=0)."""
    cm = np.bincount(k * np.asarray(y) + np.asarray(p), minlength=k * k).reshape(k, k)
    tp = np.diag(cm).astype(float)
    penyebut = cm.sum(0) + cm.sum(1)
    return np.divide(2 * tp, penyebut, out=np.zeros(k), where=penyebut > 0)


# ---- Tabel 5: hasil per fold
tab5 = pd.DataFrame({"Fold": sorted(hasil_cv["Fold"].unique())})
for m, singk in [("SVM", "SVM"), ("CNN-BiLSTM", "CNN"), ("IndoBERT", "BERT")]:
    d = hasil_cv[hasil_cv["Model"] == m].sort_values("Fold")
    assert len(d) == K_FOLD, f"Jumlah fold untuk {m} tidak lengkap."
    tab5[f"{singk} Acc."] = d["Accuracy"].values
    tab5[f"{singk} Macro F1"] = d["Macro F1"].values

print("TABEL 5 - hasil per fold")
display(tab5.round(4))
ringkas = {kol: f"{tab5[kol].mean():.4f} ± {tab5[kol].std():.4f}" for kol in tab5.columns[1:]}
print("Mean ± SD:", ringkas)

# ---- Tabel 6: F1 per kelas (out-of-fold)
baris6 = []
for m in NAMA_MODEL:
    f1k = f1_per_kelas(y_cv, prediksi_oof[m])
    baris6.append({"Model": m, **{f"F1 {n}": f for n, f in zip(NAMA_KELAS, f1k)}, "Macro F1 OOF": f1k.mean()})
tab6 = pd.DataFrame(baris6)
print("\nTABEL 6 - F1 per kelas (OOF, n =", len(y_cv), ")")
print("Jumlah per kelas:", dict(zip(NAMA_KELAS, np.bincount(y_cv, minlength=3))))
display(tab6.round(4))

# ---- Tabel 7: irisan kesalahan CNN-BiLSTM vs IndoBERT
benar_c = prediksi_oof["CNN-BiLSTM"] == y_cv
benar_b = prediksi_oof["IndoBERT"] == y_cv
tab7 = pd.DataFrame({
    "Kelompok prediksi": ["Salah pada kedua model", "Salah hanya CNN-BiLSTM", "Salah hanya IndoBERT",
                          "Benar pada kedua model", "Salah minimal satu model"],
    "Jumlah komentar": [int((~benar_c & ~benar_b).sum()), int((~benar_c & benar_b).sum()),
                        int((benar_c & ~benar_b).sum()), int((benar_c & benar_b).sum()),
                        int((~benar_c | ~benar_b).sum())],
})
print("\nTABEL 7 - irisan kesalahan")
display(tab7)

# Konsistensi dengan tabel 'analisis', jika sudah dibuat
if "analisis" in globals():
    assert (analisis["salah_cnn"].values == ~benar_c).all(), "analisis.salah_cnn tidak cocok dengan prediksi_oof."
    assert (analisis["salah_bert"].values == ~benar_b).all(), "analisis.salah_bert tidak cocok dengan prediksi_oof."
    print("Tabel 'analisis' konsisten dengan prediksi_oof.")



# ---- Simpan artefak
hasil_cv.to_csv(f"{OUT_DIR}/hasil_cv_per_fold.csv", index=False)
tab5.to_csv(f"{OUT_DIR}/tabel5_per_fold.csv", index=False)
tab6.to_csv(f"{OUT_DIR}/tabel6_f1_per_kelas_oof.csv", index=False)
tab7.to_csv(f"{OUT_DIR}/tabel7_irisan_kesalahan.csv", index=False)
oof_df = data_cv[["Komentar", "text", "label", "y"]].copy().reset_index(drop=True)
oof_df["pred_svm"] = [id2label[i] for i in prediksi_oof["SVM"]]
oof_df["pred_cnn"] = [id2label[i] for i in prediksi_oof["CNN-BiLSTM"]]
oof_df["pred_bert"] = [id2label[i] for i in prediksi_oof["IndoBERT"]]
oof_df.to_csv(f"{OUT_DIR}/oof_predictions.csv", index=False, encoding="utf-8-sig")
print("\nArtefak tersimpan di folder:", OUT_DIR)

## Confusion matrix dan metrik per kelas

In [ ]:
# =============================================
# CONFUSION MATRIX + PRECISION/RECALL/F1 PER KELAS
# =============================================
fig, axes = plt.subplots(1, len(NAMA_MODEL), figsize=(5 * len(NAMA_MODEL), 4.2))
baris_prf = []
for ax, m in zip(axes, NAMA_MODEL):
    p = prediksi_oof[m]
    cm = confusion_matrix(y_cv, p, labels=[0, 1, 2])
    cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    ax.imshow(cm_norm, vmin=0, vmax=1, cmap="Blues")
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{cm[i, j]}\n({cm_norm[i, j] * 100:.0f}%)", ha="center", va="center",
                    color="white" if cm_norm[i, j] > 0.5 else "black", fontsize=9)
    ax.set_xticks(range(3)); ax.set_xticklabels(NAMA_KELAS)
    ax.set_yticks(range(3)); ax.set_yticklabels(NAMA_KELAS)
    ax.set_xlabel("Prediksi"); ax.set_ylabel("Label manual")
    ax.set_title(f"{m} (out-of-fold, n={len(y_cv)})")

    pr, rc, f1, sup = precision_recall_fscore_support(y_cv, p, labels=[0, 1, 2], zero_division=0)
    for k in range(3):
        baris_prf.append({"Model": m, "Kelas": NAMA_KELAS[k], "Precision": pr[k], "Recall": rc[k],
                          "F1": f1[k], "Support": int(sup[k])})
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/confusion_matrix_oof.png", dpi=300)
plt.show()

tab_prf = pd.DataFrame(baris_prf)
display(tab_prf.round(4))
tab_prf.to_csv(f"{OUT_DIR}/precision_recall_f1_per_kelas_oof.csv", index=False)

## Analisis kesalahan

In [ ]:
# ============================================
# DATA ANALISIS KESALAHAN
# ============================================

analisis = data_cv[
    ["Komentar", "text", "label", "y"]
].copy().reset_index(drop=True)

analisis["pred_svm"] = prediksi_oof["SVM"]
analisis["pred_cnn"] = prediksi_oof["CNN-BiLSTM"]
analisis["pred_bert"] = prediksi_oof["IndoBERT"]

analisis["salah_svm"] = analisis["y"] != analisis["pred_svm"]

analisis["salah_cnn"] = (
    analisis["y"] != analisis["pred_cnn"]
)

analisis["salah_bert"] = (
    analisis["y"] != analisis["pred_bert"]
)

analisis["pred_svm"] = analisis["pred_svm"].map(id2label)
analisis["pred_cnn"] = analisis["pred_cnn"].map(id2label)
analisis["pred_bert"] = analisis["pred_bert"].map(id2label)

# Komentar yang salah pada minimal satu model
kesalahan = analisis[
    analisis["salah_cnn"] | analisis["salah_bert"]
].copy()

# ID untuk memudahkan anotasi manual
kesalahan["id_analisis"] = kesalahan.index

# Kolom anotasi linguistik
kesalahan["pola_linguistik"] = ""
kesalahan["catatan"] = ""

print("Total data CV:", len(analisis))
print("Kesalahan SVM:", analisis["salah_svm"].sum())
print("Kesalahan CNN-BiLSTM:", analisis["salah_cnn"].sum())
print("Kesalahan IndoBERT:", analisis["salah_bert"].sum())

print(
    "Salah pada kedua model:",
    (analisis["salah_cnn"] & analisis["salah_bert"]).sum()
)

print(
    "Komentar perlu dianalisis:",
    len(kesalahan)
)

display(
    kesalahan[
        [
            "id_analisis",
            "Komentar",
            "label",
            "pred_cnn",
            "pred_bert",
            "salah_cnn",
            "salah_bert"
        ]
    ].head(30)
)

In [ ]:
# =============================================
# KELOMPOK KESALAHAN
# =============================================

# Menggunakan dataframe 'analisis' dari sel sebelumnya

analisis["kelompok_kesalahan"] = "Benar Keduanya"

analisis.loc[
    analisis["salah_cnn"] & analisis["salah_bert"],
    "kelompok_kesalahan"
] = "Salah Keduanya"

analisis.loc[
    analisis["salah_cnn"] & ~analisis["salah_bert"],
    "kelompok_kesalahan"
] = "Salah CNN Saja"

analisis.loc[
    ~analisis["salah_cnn"] & analisis["salah_bert"],
    "kelompok_kesalahan"
] = "Salah IndoBERT Saja"

print("=" * 60)
print("DISTRIBUSI KELOMPOK KESALAHAN")
print("=" * 60)

print(
    analisis["kelompok_kesalahan"]
    .value_counts()
)

# Ambil 10 komentar dari tiap kelompok kesalahan
kelompok = [
    "Salah Keduanya",
    "Salah CNN Saja",
    "Salah IndoBERT Saja"
]

sampel_error = []

for nama in kelompok:

    bagian = analisis[
        analisis["kelompok_kesalahan"] == nama
    ]

    if len(bagian) == 0:
        continue

    sampel = bagian.sample(
        n=min(10, len(bagian)),
        random_state=SEED
    )

    sampel_error.append(sampel)

sampel_error = pd.concat(
    sampel_error
).sort_index()

# Tampilkan komentar lengkap
for idx, row in sampel_error.iterrows():

    print("\n" + "=" * 65)
    print(f"ID DATA: {idx}")
    print("KELOMPOK:", row["kelompok_kesalahan"])
    print("=" * 65)

    print("Komentar:", row["Komentar"])
    print("Label asli:", row["label"])
    print("Prediksi CNN:", row["pred_cnn"])
    print("Prediksi IndoBERT:", row["pred_bert"])

print("\nTotal sampel:", len(sampel_error))

In [ ]:
# =============================================
# POLA KESALAHAN: ARAH, PANJANG, CIRI TEKS
# =============================================

pola = data_cv[["Komentar", "text"]].reset_index(drop=True).copy()
pola["salah_svm"]  = prediksi_oof["SVM"] != y_cv
pola["salah_cnn"]  = prediksi_oof["CNN-BiLSTM"] != y_cv
pola["salah_bert"] = prediksi_oof["IndoBERT"]   != y_cv
pola["n_kata"]     = pola["text"].str.split().str.len()   # kata setelah preprocessing

# ---- (a) Arah kesalahan: label manual -> prediksi
baris_arah = []
for m in NAMA_MODEL:
    p = prediksi_oof[m]
    for i in range(3):
        for j in range(3):
            if i != j:
                baris_arah.append({"Model": m, "Label": NAMA_KELAS[i], "Prediksi": NAMA_KELAS[j],
                                   "Jumlah": int(((y_cv == i) & (p == j)).sum())})
tab_arah = pd.DataFrame(baris_arah)
tab_arah["% dari kesalahan model"] = (100 * tab_arah["Jumlah"]
                                      / tab_arah.groupby("Model")["Jumlah"].transform("sum"))
tab_arah = tab_arah.sort_values(["Model", "Jumlah"], ascending=[True, False])
print("ARAH KESALAHAN")
display(tab_arah.round(1))
tab_arah.to_csv(f"{OUT_DIR}/arah_kesalahan_oof.csv", index=False)

# ---- (b) Tingkat kesalahan menurut panjang komentar (Tabel 12)
pola["Panjang (kata)"] = pd.cut(pola["n_kata"], [0, 8, 15, 25, np.inf],
                                labels=["1–8", "9–15", "16–25", ">25"])
tab12 = (pola.groupby("Panjang (kata)", observed=True)
             .agg(n=("salah_cnn", "size"),
                  **{"Salah SVM (%)":        ("salah_svm", lambda s: 100 * s.mean()),
                     "Salah CNN-BiLSTM (%)": ("salah_cnn", lambda s: 100 * s.mean()),
                     "Salah IndoBERT (%)":   ("salah_bert", lambda s: 100 * s.mean())}))
print("\nTINGKAT KESALAHAN MENURUT PANJANG KOMENTAR")
display(tab12.round(1))
tab12.to_csv(f"{OUT_DIR}/tabel12_kesalahan_per_panjang.csv")

# ---- (c) Tingkat kesalahan menurut ciri teks yang terdeteksi otomatis
kata  = pola["text"].str.findall(r"[a-z]+")
NEGASI = {"tidak", "bukan", "belum", "jangan", "kurang"}
TANYA  = {"apa", "gimana", "bagaimana", "mana", "berapa", "kah", "kenapa"}
n_en = (data_cv[["text"]].merge(audit_final[["text", "Bukti Inggris"]], on="text", how="left")
        ["Bukti Inggris"].fillna("")
        .map(lambda s: len([k for k in str(s).split(",") if k.strip()])).values)

ciri = {
    "Negasi":             kata.map(lambda w: any(x in NEGASI for x in w)),
    "Pertanyaan":         pola["Komentar"].str.contains(r"\?") | kata.map(lambda w: any(x in TANYA for x in w)),
    "≥2 kata Inggris":    pd.Series(n_en >= 2),
    "Emoji":              pola["Komentar"].str.contains(r"[\U0001F300-\U0001FAFF\u2600-\u27BF]"),
    "Balasan (@mention)": pola["Komentar"].str.contains("@"),
}

baris_ciri = []
for nama, ada in ciri.items():
    ada = ada.values.astype(bool)
    r = {"Ciri": nama, "n ada": int(ada.sum()),
         "Median kata (ada)": float(np.median(pola.loc[ada, "n_kata"])),
         "Median kata (tidak)": float(np.median(pola.loc[~ada, "n_kata"]))}
    for m, kol in [("SVM", "salah_svm"), ("CNN", "salah_cnn"), ("BERT", "salah_bert")]:
        s = pola[kol].values
        tabel = [[int((s & ada).sum()),  int((~s & ada).sum())],
                 [int((s & ~ada).sum()), int((~s & ~ada).sum())]]
        r[f"% salah {m} (ada)"]   = 100 * s[ada].mean()
        r[f"% salah {m} (tidak)"] = 100 * s[~ada].mean()
        r[f"p Fisher {m}"]        = stats.fisher_exact(tabel)[1]
    baris_ciri.append(r)
tab_ciri = pd.DataFrame(baris_ciri)
print("\nTINGKAT KESALAHAN MENURUT CIRI TEKS")
display(tab_ciri.round(3))
tab_ciri.to_csv(f"{OUT_DIR}/kesalahan_per_ciri_teks.csv", index=False)

## Deskripsi data

In [ ]:
# =============================================
# DESKRIPSI DATA: PER VIDEO, BUKTI CODE-MIXING, KATA PALING SERING
# =============================================
from collections import Counter

# ---- (a) Distribusi sentimen per video
dist_video = pd.crosstab(df["Video ID"], df["label"], margins=True)[NAMA_KELAS + ["All"]]
print("DISTRIBUSI SENTIMEN PER VIDEO")
display(dist_video)
dist_video.to_csv(f"{OUT_DIR}/distribusi_per_video.csv")

# ---- (b) Ringkasan bukti code-mixing (berdasarkan kolom 'Bukti Indonesia'/'Bukti Inggris' pada dataset)
# Catatan: ini perkiraan berbasis bukti yang tercatat di dataset, bukan language identification per token.
if "audit_final" in globals():
    def daftar_bukti(s):
        if pd.isna(s) or not str(s).strip():
            return []
        return [k.strip().lower() for k in str(s).split(",") if k.strip()]

    ev_id = audit_final["Bukti Indonesia"].map(daftar_bukti)
    ev_en = audit_final["Bukti Inggris"].map(daftar_bukti)
    n_id, n_en = ev_id.map(len), ev_en.map(len)
    rasio_en = n_en / (n_id + n_en).replace(0, np.nan)

    print("\nBUKTI CODE-MIXING (n =", len(audit_final), "komentar)")
    ringkas_cm = pd.DataFrame({
        "Bukti Indonesia": n_id.describe(), "Bukti Inggris": n_en.describe(), "Rasio bukti Inggris": rasio_en.describe()
    }).round(3)
    display(ringkas_cm)
    print("Komentar dengan tepat 1 kata bukti Inggris :", int((n_en == 1).sum()), f"({(n_en == 1).mean() * 100:.1f}%)")
    print("Komentar dengan 2-3 kata bukti Inggris     :", int(n_en.between(2, 3).sum()), f"({n_en.between(2, 3).mean() * 100:.1f}%)")
    print("Komentar dengan >= 4 kata bukti Inggris    :", int((n_en >= 4).sum()), f"({(n_en >= 4).mean() * 100:.1f}%)")

    top_en = Counter(k for daftar in ev_en for k in daftar).most_common(25)
    print("\n25 kata bukti Inggris paling sering:")
    print(", ".join(f"{k} ({v})" for k, v in top_en))
    ringkas_cm.to_csv(f"{OUT_DIR}/ringkasan_bukti_codemixing.csv")
    pd.DataFrame(top_en, columns=["kata", "frekuensi"]).to_csv(f"{OUT_DIR}/kata_inggris_tersering.csv", index=False)
else:
    print("\n'audit_final' belum ada; jalankan bagian audit code-mixed (2b) terlebih dahulu.")

# ---- (c) Kandidat slang untuk memperluas kamus SLANG (sel 'kata paling sering' yang disebut di komentar kamus)
# Tinjau MANUAL. Jika kamus diperluas, seluruh notebook harus dijalankan ulang dari sel preprocessing
# (teks berubah, sehingga split dan semua hasil ikut berubah).
sudah_ada = set(SLANG) | set(SLANG.values())
hitung = Counter(kata for t in df["text"] for kata in re.findall(r"[a-z]+", t) if len(kata) <= 5)
kandidat = [(k, v) for k, v in hitung.most_common() if v >= 5 and k not in sudah_ada]
print("\nKANDIDAT SLANG/SINGKATAN (panjang <= 5 huruf, frekuensi >= 5), tinjau manual:")
print(", ".join(f"{k} ({v})" for k, v in kandidat[:120]))

## Regresi logistik ciri teks

In [ ]:
# REGRESI LOGISTIK: PENGARUH CIRI TEKS SETELAH MENGONTROL PANJANG KOMENTAR
import statsmodels.formula.api as smf
nama_var = {"Negasi": "negasi", "Pertanyaan": "tanya", "≥2 kata Inggris": "inggris2",
            "Emoji": "emoji", "Balasan (@mention)": "balasan"}
dfm = pd.DataFrame({v: ciri[k].values.astype(int) for k, v in nama_var.items()})
dfm["log_kata"] = np.log(pola["n_kata"].values)
baris_lr = []
for m, kol in [("SVM", "salah_svm"), ("CNN-BiLSTM", "salah_cnn"), ("IndoBERT", "salah_bert")]:
    dfm["salah"] = pola[kol].values.astype(int)
    hasil = smf.logit("salah ~ negasi + tanya + inggris2 + emoji + balasan + log_kata", data=dfm).fit(disp=0)
    ci = hasil.conf_int()
    print(f"{m}: p model keseluruhan (LLR) = {hasil.llr_pvalue:.4g}")
    for v in list(nama_var.values()) + ["log_kata"]:
        baris_lr.append({"Model": m, "Variabel": v, "OR": np.exp(hasil.params[v]),
                         "CI 2.5%": np.exp(ci.loc[v, 0]), "CI 97.5%": np.exp(ci.loc[v, 1]), "p": hasil.pvalues[v]})
tab_lr = pd.DataFrame(baris_lr)
print("\nREGRESI LOGISTIK (dikontrol panjang komentar)")
display(tab_lr.round(3))
tab_lr.to_csv(f"{OUT_DIR}/regresi_logistik_ciri.csv", index=False)

## Uji statistik lanjutan

In [ ]:
# =============================================
# UJI STATISTIK: IK TERKOREKSI, KOREKSI HOLM, BOOTSTRAP KELAS NEGATIF
# =============================================
from sklearn.metrics import f1_score, recall_score

# (a) Setiap skema vs tanpa penanganan (seed 42, 5 fold, Nadeau-Bengio) + Holm atas 8 uji
rows = []
for m in NAMA_MODEL:
    dasar = hasil_imb[(hasil_imb.Model == m) & (hasil_imb.Metode == "Tanpa penanganan")].sort_values("Fold")["Macro F1"].values
    for metode in METODE_MODEL[m]:
        if metode == "Tanpa penanganan":
            continue
        v = hasil_imb[(hasil_imb.Model == m) & (hasil_imb.Metode == metode)].sort_values("Fold")["Macro F1"].values
        rows.append({"Model": m, "Skema": metode, **uji_terkoreksi(v - dasar, 1)})
uji_skema = pd.DataFrame(rows); uji_skema["p Holm"] = holm(uji_skema["p"])
print("SKEMA vs TANPA PENANGANAN (seed 42, Macro F1)"); display(uji_skema.round(4))
uji_skema.to_csv(f"{OUT_DIR}/uji_skema_vs_tanpa_holm.csv", index=False)

# (b) Perbandingan model dengan skema terpilih (5 seed x 5 fold) + Holm per metrik
ms = pd.read_csv(f"{OUT_DIR}/cv_multi_seed_per_fold.csv")
rows = []
for met in ["Macro F1", "Accuracy"]:
    pv = ms.pivot_table(index=["Seed", "Fold"], columns="Model", values=met)
    sub = []
    for a_, b_ in [("IndoBERT", "CNN-BiLSTM"), ("IndoBERT", "SVM"), ("SVM", "CNN-BiLSTM")]:
        d = (pv[a_] - pv[b_]).values
        sub.append({"Pasangan": f"{a_} vs {b_}", "Metrik": met, "Unggul fold": int((d > 0).sum()), **uji_terkoreksi(d, 5)})
    sub = pd.DataFrame(sub); sub["p Holm"] = holm(sub["p"]); rows.append(sub)
uji_terpilih = pd.concat(rows, ignore_index=True)
print("\nPERBANDINGAN MODEL, SKEMA TERPILIH (5 seed x 5 fold)"); display(uji_terpilih.round(4))
uji_terpilih.to_csv(f"{OUT_DIR}/uji_model_skema_terpilih_holm.csv", index=False)

# (c) Sensitivitas terhadap n1: n1 = ukuran data latih internal (sekitar 1.037)
n1_asli = N_LATIH
N_LATIH = len(data_cv) - N_UJI - round(0.10 * (len(data_cv) - N_UJI))
cek = pd.DataFrame([{"Uji": f"{r.Model} {r.Skema}", "p (n1 = 1152)": r.p,
                     "p (n1 internal)": uji_terkoreksi(hasil_imb[(hasil_imb.Model == r.Model) & (hasil_imb.Metode == r.Skema)].sort_values("Fold")["Macro F1"].values
                                                      - hasil_imb[(hasil_imb.Model == r.Model) & (hasil_imb.Metode == "Tanpa penanganan")].sort_values("Fold")["Macro F1"].values, 1)["p"]}
                    for r in uji_skema.itertuples()])
N_LATIH = n1_asli
print(f"\nSensitivitas n1 (n1 internal = {len(data_cv) - N_UJI - round(0.10 * (len(data_cv) - N_UJI)):.0f})"); display(cek.round(4))

# (d) Regresi logistik: Holm atas 18 koefisien
tab_lr["p Holm"] = holm(tab_lr["p"])
print("\nREGRESI LOGISTIK + HOLM"); display(tab_lr.round(4))
tab_lr.to_csv(f"{OUT_DIR}/regresi_logistik_ciri_holm.csv", index=False)

# (e) Bootstrap 95% F1 dan recall kelas Negatif (prediksi gabungan 5-fold CV, seed 42)
def f1_neg(yt, yp): return f1_score(yt, yp, labels=[0], average=None, zero_division=0)[0]
def rec_neg(yt, yp): return recall_score(yt, yp, labels=[0], average=None, zero_division=0)[0]
rng = np.random.default_rng(42); B = 5000
idx = [rng.integers(0, len(y_cv), len(y_cv)) for _ in range(B)]
P = dict(prediksi_oof)
with open(os.path.join(OUT_DIR, "indobertweet", "Tanpa_penanganan.pkl"), "rb") as fh:
    P["IndoBERTweet"] = pickle.load(fh)["oof"]
rows = []
for m, p in P.items():
    f = [f1_neg(y_cv[i], p[i]) for i in idx]; r = [rec_neg(y_cv[i], p[i]) for i in idx]
    rows.append({"Model": m, "F1 Negatif": f1_neg(y_cv, p), "F1 IK bawah": np.percentile(f, 2.5), "F1 IK atas": np.percentile(f, 97.5),
                 "Recall Negatif": rec_neg(y_cv, p), "Recall IK bawah": np.percentile(r, 2.5), "Recall IK atas": np.percentile(r, 97.5)})
dasar = oof_imb[("IndoBERT", "Tanpa penanganan")]
for metode in ["Focal loss", "Class weighting"]:
    alt = oof_imb[("IndoBERT", metode)]
    df_ = [f1_neg(y_cv[i], alt[i]) - f1_neg(y_cv[i], dasar[i]) for i in idx]
    dr = [rec_neg(y_cv[i], alt[i]) - rec_neg(y_cv[i], dasar[i]) for i in idx]
    rows.append({"Model": f"IndoBERT {metode} - tanpa penanganan", "F1 Negatif": f1_neg(y_cv, alt) - f1_neg(y_cv, dasar),
                 "F1 IK bawah": np.percentile(df_, 2.5), "F1 IK atas": np.percentile(df_, 97.5),
                 "Recall Negatif": rec_neg(y_cv, alt) - rec_neg(y_cv, dasar),
                 "Recall IK bawah": np.percentile(dr, 2.5), "Recall IK atas": np.percentile(dr, 97.5)})
boot = pd.DataFrame(rows)
print("\nBOOTSTRAP KELAS NEGATIF"); display(boot.round(4))
boot.to_csv(f"{OUT_DIR}/bootstrap_kelas_negatif.csv", index=False)

# Metrik per kelas IndoBERTweet (seed 42)
pr, rc, f1, sup = precision_recall_fscore_support(y_cv, P["IndoBERTweet"], labels=[0, 1, 2], zero_division=0)
print("\nIndoBERTweet per kelas"); display(pd.DataFrame({"Kelas": NAMA_KELAS, "Precision": pr, "Recall": rc, "F1": f1, "Support": sup}).round(4))
print(pd.DataFrame(confusion_matrix(y_cv, P["IndoBERTweet"]), index=NAMA_KELAS, columns=NAMA_KELAS))

## Kesepakatan dengan annotator kedua

In [ ]:
# =============================================
# KESEPAKATAN DENGAN ANNOTATOR KEDUA
# =============================================
# label_annotator2.xlsx: 320 komentar (100 Negatif, 120 Netral, 100 Positif menurut annotator 1).
# Kolom "Baris di fix_revisibaru.xlsx" = nomor baris Excel pada sheet "Data Sentimen".
a2 = pd.read_excel(ANOTASI2_PATH)
mentah = pd.read_excel(DATA_PATH, sheet_name="Data Sentimen")
a2["Komentar"] = mentah["Komentar"].values[a2["Baris di fix_revisibaru.xlsx"].values - 2]
assert (mentah["Sentimen manual"].values[a2["Baris di fix_revisibaru.xlsx"].values - 2] == a2["Label Annotator 1"]).all()
A1, A2, L = a2["Label Annotator 1"], a2["Label Annotator 2"], a2["Label LLM"]
KELAS = ["Negatif", "Netral", "Positif"]

def kappa(x, y, w=None):
    return cohen_kappa_score(x, y, sample_weight=w), np.average(np.asarray(x) == np.asarray(y), weights=w)

k, ag = kappa(A1, A2); print(f"Annotator 1 vs 2: kappa = {k:.4f}, kesepakatan = {ag:.2%} (n = {len(a2)})")
rng = np.random.default_rng(0)
bs = [cohen_kappa_score(A1.values[i], A2.values[i]) for i in (rng.integers(0, len(a2), len(a2)) for _ in range(5000))]
print("IK bootstrap 95%:", np.percentile(bs, [2.5, 97.5]).round(4))
pop = mentah.loc[mentah["Komentar"].notna(), "Sentimen manual"].value_counts(normalize=True)
w = A1.map(pop / A1.value_counts(normalize=True)).values
k, ag = kappa(A1, A2, w); print(f"Dibobot ke proporsi populasi: kappa = {k:.4f}, kesepakatan = {ag:.2%}")
k, ag = kappa(A2, L); print(f"Annotator 2 vs LLM: kappa = {k:.4f}")
print("\\nMatriks annotator 1 (baris) vs annotator 2 (kolom)")
display(pd.DataFrame(confusion_matrix(A1, A2, labels=KELAS), index=KELAS, columns=KELAS))

# Tanpa komentar pertanyaan/permintaan
KATA_TANYA = {"apa", "apakah", "gimana", "bagaimana", "mana", "berapa", "kah", "kenapa"}
POLA_MINTA = r"\b(minta|tolong|request|rekomendasi\w*|rekomen\w*|saran\w*|mohon|review\w* dong|bahas|reviewin|riview\w*)\b"
def tanya_minta(s):
    t = s.astype(str).str.lower()
    return (t.str.contains(r"\?") | t.str.findall(r"[a-z]+").map(lambda ws: any(x in KATA_TANYA for x in ws))
            | t.str.contains(POLA_MINTA))
q = tanya_minta(a2["Komentar"])
k, _ = kappa(A1[~q], A2[~q]); print(f"\nAnnotator 1 vs 2 tanpa pertanyaan/permintaan: kappa = {k:.4f} (n = {int((~q).sum())})")
s = mentah[mentah["Komentar"].notna()]
qs = tanya_minta(s["Komentar"])
k, ag = kappa(s["Sentimen manual"], s["Sentimen"]); print(f"Annotator 1 vs LLM (semua): kappa = {k:.4f}, kesepakatan = {ag:.2%} (n = {len(s)})")
k, ag = kappa(s.loc[~qs, "Sentimen manual"], s.loc[~qs, "Sentimen"])
print(f"Annotator 1 vs LLM tanpa pertanyaan/permintaan: kappa = {k:.4f}, kesepakatan = {ag:.2%} (n = {int((~qs).sum())})")

## Validasi filter code-mixing

In [ ]:
# =============================================
# VALIDASI FILTER CODE-MIXING
# =============================================
# validasi_filter.xlsx: 200 komentar lolos filter + 200 tidak lolos, diperiksa manual tanpa melihat status filter.
vf = pd.read_excel(VALIDASI_FILTER_PATH)
N_LOLOS, N_TIDAK = 1600, 5884        # populasi komentar unik yang lolos / tidak lolos filter
lolos = vf["Status filter"] == "Lolos"
cm_ = vf["Indonesia + Inggris (manual)"] == "Ya"
serapan = vf["Hanya kata serapan (manual)"].fillna("Tidak") == "Ya"
print(pd.crosstab(vf["Status filter"], vf["Indonesia + Inggris (manual)"]))
a, n_f = int((lolos & cm_).sum()), int(lolos.sum())
b, n_nf = int((~lolos & cm_).sum()), int((~lolos).sum())

def wilson(x, n, z=1.96):
    p = x / n; den = 1 + z ** 2 / n
    c = (p + z ** 2 / (2 * n)) / den; h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / den
    return c - h, c + h

def prf(a, b):
    P = a / n_f; q = b / n_nf
    tp, fn = P * N_LOLOS, q * N_TIDAK
    R = tp / (tp + fn)
    return P, R, 2 * P * R / (P + R)

P, R, F = prf(a, b)
rng = np.random.default_rng(0)
bt = np.array([prf(rng.binomial(n_f, a / n_f), rng.binomial(n_nf, b / n_nf)) for _ in range(5000)])
print(f"\nPrecision = {a}/{n_f} = {P:.3f}  IK Wilson {wilson(a, n_f)[0]:.3f}-{wilson(a, n_f)[1]:.3f}")
print(f"Recall    = {R:.3f}  IK bootstrap {np.percentile(bt[:, 1], 2.5):.3f}-{np.percentile(bt[:, 1], 97.5):.3f}")
print(f"F1        = {F:.3f}  IK bootstrap {np.percentile(bt[:, 2], 2.5):.3f}-{np.percentile(bt[:, 2], 97.5):.3f}")
s_ = int((lolos & cm_ & serapan).sum())
print(f"Lolos, code-mixed, tetapi hanya kata serapan: {s_}/{n_f} = {s_ / n_f:.1%}  IK Wilson {wilson(s_, n_f)[0]:.3f}-{wilson(s_, n_f)[1]:.3f}")

## Kemas hasil

In [ ]:
# =============================================
# KEMAS SEMUA HASIL
# =============================================
import shutil
shutil.make_archive("/content/hasil_penelitian", "zip", OUT_DIR)
try:
    from google.colab import files
    files.download("/content/hasil_penelitian.zip")
except Exception:
    print("Unduh hasil_penelitian.zip dari panel file Colab.")